# Machine Learning Based Optimal Control of Delayed Sobolev-Type Fractional Evolution Systems of Order (1,2) under Seismic Disturbances
**Numerical validation: ABM scheme, fPINN and neural network controller**

M. Siddiq and A. U. K. Niazi, Department of Mathematics and Statistics, The University of Lahore, Sargodha

This notebook reproduces all figures and tables of Section 6 of the paper.
It runs on **Google Colab**, **Kaggle** or a local **Jupyter** installation. A CPU is enough (no GPU needed).

**How to run:** run all cells (Colab: `Runtime → Run all`; Kaggle: `Run All`).
Total time is about 60–75 minutes. The longest steps are Step 4 (≈18 min) and Step 6 (≈30 min).

| Step | Script | Paper output |
|---|---|---|
| 3 | `fig0_model.py` | Figure 1 (model problem) |
| 4 | `numerics.py` | Figures 2, 3; Table 3 (smooth forcing) |
| 5 | `fig3_order.py` | Figure 4 (effect of the order ϱ) |
| 6 | `numerics_elcentro.py` | Figures 6, 7; Tables 4, 5, 6 (El Centro) |
| 7 | `fpinn_elcentro.py` | Figure 5; Table 3 (El Centro column) |
| 8 | `export_data.py` | CSV data of all tables and figures |
| 8b | `revision_checks.py` | Grid convergence and other checks of the revision (Tables 5, 6, 8) |

## Step 1. Set up folders and check the libraries

In [ ]:
import os
# Working folder: /kaggle/working on Kaggle, /content on Colab, current folder elsewhere
if os.path.isdir('/kaggle/working'):
    os.chdir('/kaggle/working')
elif os.path.isdir('/content'):
    os.chdir('/content')
BASE = os.path.abspath('project')
for d in ['code', 'data/raw', 'data/processed', 'results']:
    os.makedirs(os.path.join(BASE, d), exist_ok=True)
import numpy, torch, matplotlib
print('Project folder:', BASE)
print('numpy', numpy.__version__, '| torch', torch.__version__, '| matplotlib', matplotlib.__version__)

## Step 2. Write the El Centro record and the Python scripts
The 1940 El Centro NS ground acceleration (time [s], acceleration [g]; source: Vibrationdata, T. Irvine) is written below, so no upload is needed.

In [ ]:
%%writefile project/data/raw/elcentro_NS.txt
0.0000000e+000 -1.4275799e-003
2.0000000e-002 -1.1012760e-002
4.0000000e-002 -1.0298970e-002
6.0000000e-002 -8.9733599e-003
8.0000000e-002 -9.6871497e-003
1.0000000e-001 -1.2236400e-002
1.2000000e-001 -1.4479739e-002
1.4000000e-001 -1.3052160e-002
1.6000000e-001 -1.1216700e-002
1.8000000e-001 -8.6674497e-003
2.0000000e-001 -8.6674497e-003
2.2000000e-001 -1.3358070e-002
2.4000000e-001 -1.7946720e-002
2.6000000e-001 -1.9782179e-002
2.8000000e-001 -1.6519140e-002
3.0000000e-001 -1.4683679e-002
3.2000000e-001 -1.1012760e-002
3.4000000e-001 -8.3615395e-003
3.6000000e-001 -4.2827397e-003
3.8000000e-001 -6.7300197e-003
4.0000000e-001 -1.3358070e-002
4.2000000e-001 -1.9374299e-002
4.4000000e-001 -1.9986120e-002
4.6000000e-001 -6.7300197e-003
4.8000000e-001 3.0590999e-003
5.0000000e-001 1.4377770e-002
5.2000000e-001 -4.9965299e-003
5.4000000e-001 -1.3052160e-002
5.6000000e-001 -1.4683679e-002
5.8000000e-001 -2.0699909e-002
6.0000000e-001 -2.6512199e-002
6.2000000e-001 -3.3140249e-002
6.4000000e-001 -3.1202819e-002
6.6000000e-001 -1.7538840e-002
6.8000000e-001 -2.0088090e-002
7.0000000e-001 -1.6621109e-002
7.2000000e-001 -1.6723079e-002
7.4000000e-001 -6.8319896e-003
7.6000000e-001 2.5492499e-003
7.8000000e-001 1.5295499e-002
8.0000000e-001 2.4064920e-002
8.2000000e-001 2.5696440e-002
8.4000000e-001 3.4261917e-002
8.6000000e-001 4.7212108e-002
8.8000000e-001 5.0169239e-002
9.0000000e-001 4.2725430e-002
9.2000000e-001 3.6607230e-002
9.4000000e-001 2.7633869e-002
9.6000000e-001 2.3962949e-002
9.8000000e-001 3.4567830e-002
1.0000000e+000 4.2011639e-002
1.0200000e+000 5.4044098e-002
1.0400000e+000 6.5158829e-002
1.0600000e+000 7.4642034e-002
1.0800000e+000 6.6484435e-002
1.1000000e+000 6.1080029e-002
1.1200000e+000 4.0787999e-002
1.1400000e+000 4.0787999e-002
1.1600000e+000 6.4241100e-003
1.1800000e+000 -5.2514548e-002
1.2000000e+000 -8.0250384e-002
1.2200000e+000 -6.1487907e-002
1.2400000e+000 -4.9353480e-002
1.2600000e+000 -2.5492499e-002
1.2800000e+000 -6.0162299e-003
1.3000000e+000 1.3663979e-002
1.3200000e+000 3.1406758e-002
1.3400000e+000 5.0883030e-002
1.3600000e+000 7.2398697e-002
1.3800000e+000 1.0146015e-001
1.4000000e+000 1.2440340e-001
1.4200000e+000 1.5601409e-001
1.4400000e+000 1.4785649e-001
1.4600000e+000 1.1828520e-001
1.4800000e+000 9.5341947e-002
1.5000000e+000 9.0957234e-002
1.5200000e+000 9.4424215e-002
1.5400000e+000 8.5552829e-002
1.5600000e+000 9.1874965e-002
1.5800000e+000 1.0125621e-001
1.6000000e+000 1.2338370e-001
1.6200000e+000 3.3446158e-002
1.6400000e+000 -1.5091559e-001
1.6600000e+000 -2.1107789e-001
1.6800000e+000 -2.0292029e-001
1.7000000e+000 -2.0699909e-001
1.7200000e+000 -1.8558539e-001
1.7400000e+000 -1.7640809e-001
1.7600000e+000 -1.7844749e-001
1.7800000e+000 -1.7844749e-001
1.8000000e+000 -1.8456569e-001
1.8200000e+000 -1.6621109e-001
1.8400000e+000 -1.3765950e-001
1.8600000e+000 -1.1114730e-001
1.8800000e+000 -7.9740534e-002
1.9000000e+000 -4.3745130e-002
1.9200000e+000 -1.7334900e-003
1.9400000e+000 3.6709199e-002
1.9600000e+000 8.0046447e-002
1.9800000e+000 1.1828520e-001
2.0000000e+000 1.6315199e-001
2.0200000e+000 1.9986119e-001
2.0400000e+000 2.4574769e-001
2.0600000e+000 2.7837809e-001
2.0800000e+000 3.0998879e-001
2.1000000e+000 3.2630399e-001
2.1200000e+000 3.4873739e-001
2.1400000e+000 2.8755539e-001
2.1600000e+000 2.3657039e-001
2.1800000e+000 -1.2236400e-001
2.2000000e+000 -2.4166889e-001
2.2200000e+000 -1.6723079e-001
2.2400000e+000 -1.9068389e-001
2.2600000e+000 -1.1216700e-001
2.2800000e+000 -7.6783410e-002
2.3000000e+000 -1.7640809e-002
2.3200000e+000 1.1522610e-002
2.3400000e+000 5.4350007e-002
2.3600000e+000 9.1263147e-002
2.3800000e+000 1.2134430e-001
2.4000000e+000 1.7946719e-001
2.4200000e+000 5.8734716e-002
2.4400000e+000 -2.6818109e-001
2.4600000e+000 -1.5805349e-001
2.4800000e+000 -1.7640809e-001
2.5000000e+000 -1.0298970e-001
2.5200000e+000 -5.9040630e-002
2.5400000e+000 2.4166890e-002
2.5600000e+000 -6.8319898e-002
2.5800000e+000 -2.0190059e-001
2.6000000e+000 -1.6723079e-001
2.6200000e+000 -1.7232929e-001
2.6400000e+000 -1.5091559e-001
2.6600000e+000 -1.2542310e-001
2.6800000e+000 -1.0197000e-001
2.7000000e+000 -7.6579466e-002
2.7200000e+000 -5.3330307e-002
2.7400000e+000 -2.7633869e-002
2.7600000e+000 -4.4866799e-003
2.7800000e+000 1.9170359e-002
2.8000000e+000 -9.6871497e-003
2.8200000e+000 -4.4153008e-002
2.8400000e+000 -8.5450860e-002
2.8600000e+000 -9.6973465e-002
2.8800000e+000 -7.3010516e-002
2.9000000e+000 -6.1080029e-002
2.9200000e+000 -3.4057980e-002
2.9400000e+000 -1.1012760e-002
2.9600000e+000 1.8864449e-002
2.9800000e+000 4.2827399e-002
3.0000000e+000 6.8625811e-002
3.0200000e+000 -9.8910895e-003
3.0400000e+000 -3.7932839e-002
3.0600000e+000 -4.0787999e-003
3.0800000e+000 1.1216700e-003
3.1000000e+000 3.5077680e-002
3.1200000e+000 5.7613048e-002
3.1400000e+000 9.0039510e-002
3.1600000e+000 1.1522610e-001
3.1800000e+000 1.3867920e-001
3.2000000e+000 2.2331429e-002
3.2200000e+000 2.4574770e-002
3.2400000e+000 6.9645511e-002
3.2600000e+000 7.0257329e-002
3.2800000e+000 1.3460040e-001
3.3000000e+000 1.3765950e-001
3.3200000e+000 2.0801879e-001
3.3400000e+000 -9.4934065e-002
3.3600000e+000 -1.3358070e-001
3.3800000e+000 -7.0563234e-002
3.4000000e+000 -5.5675617e-002
3.4200000e+000 7.3418396e-003
3.4400000e+000 6.8829748e-002
3.4600000e+000 -1.0910790e-001
3.4800000e+000 -1.5193529e-001
3.5000000e+000 -1.0910790e-001
3.5200000e+000 -1.1828520e-001
3.5400000e+000 -7.7701134e-002
3.5600000e+000 -5.7001230e-002
3.5800000e+000 -2.1923549e-002
3.6000000e+000 -1.2848220e-002
3.6200000e+000 -6.8727779e-002
3.6400000e+000 -3.3038280e-002
3.6600000e+000 -3.4363890e-002
3.6800000e+000 -1.1114729e-002
3.7000000e+000 1.7334900e-003
3.7200000e+000 3.0489029e-002
3.7400000e+000 4.9761357e-002
3.7600000e+000 6.1997757e-002
3.7800000e+000 2.2637340e-002
3.8000000e+000 -3.2630399e-003
3.8200000e+000 -2.4982649e-002
3.8400000e+000 7.8516895e-003
3.8600000e+000 2.1515670e-002
3.8800000e+000 5.7918957e-002
3.9000000e+000 8.4227216e-002
3.9200000e+000 1.2338370e-001
3.9400000e+000 1.5091559e-001
3.9600000e+000 1.7742779e-001
3.9800000e+000 4.2929367e-002
4.0000000e+000 2.9571300e-003
4.0200000e+000 2.6410229e-002
4.0400000e+000 2.9877208e-002
4.0600000e+000 -5.6083498e-003
4.0800000e+000 -1.4989589e-002
4.1000000e+000 1.4581710e-002
4.1200000e+000 2.1005820e-002
4.1400000e+000 5.0883030e-002
4.1600000e+000 6.5770648e-002
4.1800000e+000 9.7585284e-002
4.2000000e+000 1.1522610e-001
4.2200000e+000 1.4785649e-001
4.2400000e+000 1.6621109e-001
4.2600000e+000 1.9884149e-001
4.2800000e+000 1.8966419e-001
4.3000000e+000 2.0190059e-001
4.3200000e+000 1.8048689e-001
4.3400000e+000 1.2746250e-001
4.3600000e+000 -1.2338370e-001
4.3800000e+000 -5.5267739e-002
4.4000000e+000 -3.9156480e-002
4.4200000e+000 -3.1712669e-002
4.4400000e+000 -1.1420640e-001
4.4600000e+000 -1.6927019e-001
4.4800000e+000 -2.5084619e-001
4.5000000e+000 -2.0699909e-001
4.5200000e+000 -1.8762479e-001
4.5400000e+000 -1.3460040e-001
4.5600000e+000 -9.7891197e-002
4.5800000e+000 -3.3140249e-002
4.6000000e+000 1.5703379e-002
4.6200000e+000 8.3207516e-002
4.6400000e+000 1.3460040e-001
4.6600000e+000 1.8558539e-001
4.6800000e+000 -5.9142600e-003
4.7000000e+000 -1.7232929e-002
4.7200000e+000 2.9061449e-002
4.7400000e+000 4.5580589e-002
4.7600000e+000 1.0023651e-001
4.7800000e+000 1.4479739e-001
4.8000000e+000 1.8864449e-001
4.8200000e+000 2.5084619e-001
4.8400000e+000 1.7232929e-001
4.8600000e+000 -1.4071860e-001
4.8800000e+000 -1.0186803e-001
4.9000000e+000 -1.1114730e-001
4.9200000e+000 -9.2486784e-002
4.9400000e+000 -4.7823930e-002
4.9600000e+000 -1.2746250e-001
4.9800000e+000 -2.1515669e-001
5.0000000e+000 -1.6519139e-001
5.0200000e+000 -1.7232929e-001
5.0400000e+000 -1.3358070e-001
5.0600000e+000 -1.1318670e-001
5.0800000e+000 -7.8822810e-002
5.1000000e+000 -5.2004698e-002
5.1200000e+000 -5.5471680e-002
5.1400000e+000 -1.2236400e-001
5.1600000e+000 -1.2338370e-001
5.1800000e+000 -1.1828520e-001
5.2000000e+000 -1.1726550e-001
5.2200000e+000 -7.3112484e-002
5.2400000e+000 -5.5675617e-002
5.2600000e+000 6.5260799e-003
5.2800000e+000 -8.1983879e-002
5.3000000e+000 -1.6621109e-001
5.3200000e+000 -8.7592229e-002
5.3400000e+000 -9.7993165e-002
5.3600000e+000 -4.0380117e-002
5.3800000e+000 -1.4989589e-002
5.4000000e+000 3.2528428e-002
5.4200000e+000 6.6076561e-002
5.4400000e+000 8.9325715e-002
5.4600000e+000 4.8129839e-002
5.4800000e+000 2.0190059e-002
5.5000000e+000 -2.7531900e-003
5.5200000e+000 2.9775240e-002
5.5400000e+000 4.5376648e-002
5.5600000e+000 8.0046447e-002
5.5800000e+000 1.0502910e-001
5.6000000e+000 1.3765950e-001
5.6200000e+000 1.6417169e-001
5.6400000e+000 1.8966419e-001
5.6600000e+000 1.3052160e-001
5.6800000e+000 6.5260798e-002
5.7000000e+000 2.0801879e-002
5.7200000e+000 3.2018579e-002
5.7400000e+000 3.8034808e-002
5.7600000e+000 5.0577117e-002
5.7800000e+000 2.3962949e-002
5.8000000e+000 -8.5654793e-003
5.8200000e+000 -1.7130959e-002
5.8400000e+000 -1.1522610e-002
5.8600000e+000 -2.3351129e-002
5.8800000e+000 -2.5288558e-002
5.9000000e+000 -1.6009289e-002
5.9200000e+000 -7.0359299e-003
5.9400000e+000 1.4989589e-002
5.9600000e+000 3.8646630e-002
5.9800000e+000 5.9040630e-002
6.0000000e+000 2.6002349e-002
6.0200000e+000 -4.1807698e-003
6.0400000e+000 -4.3643158e-002
6.0600000e+000 -1.3562010e-002
6.0800000e+000 9.6871497e-003
6.1000000e+000 2.3453099e-002
6.1200000e+000 -1.3154129e-002
6.1400000e+000 -5.0984998e-003
6.1600000e+000 8.1575997e-003
6.1800000e+000 2.1413699e-002
6.2000000e+000 3.8748599e-002
6.2200000e+000 5.2004698e-002
6.2400000e+000 1.6009289e-002
6.2600000e+000 -3.2630399e-003
6.2800000e+000 -1.1318670e-002
6.3000000e+000 5.0984998e-004
6.3200000e+000 7.7497196e-003
6.3400000e+000 3.5689499e-003
6.3600000e+000 -9.6871497e-003
6.3800000e+000 -3.6709198e-003
6.4000000e+000 -1.6315200e-003
6.4200000e+000 3.8748598e-003
6.4400000e+000 8.6674497e-003
6.4600000e+000 -5.7103197e-003
6.4800000e+000 -3.0998879e-002
6.5000000e+000 -4.2929367e-002
6.5200000e+000 -2.4880679e-002
6.5400000e+000 -2.4064920e-002
6.5600000e+000 -1.8048690e-002
6.5800000e+000 -1.3154129e-002
6.6000000e+000 -1.8354599e-003
6.6200000e+000 2.0699909e-002
6.6400000e+000 -1.1012760e-002
6.6600000e+000 -9.2792701e-003
6.6800000e+000 -3.4669800e-003
6.7000000e+000 -1.0808820e-002
6.7200000e+000 -1.1318670e-002
6.7400000e+000 -1.0095029e-002
6.7600000e+000 -2.0394000e-004
6.7800000e+000 7.4438099e-003
6.8000000e+000 2.3962949e-002
6.8200000e+000 3.6199349e-002
6.8400000e+000 7.1888848e-002
6.8600000e+000 7.9434629e-002
6.8800000e+000 1.8762479e-002
6.9000000e+000 -2.6818108e-002
6.9200000e+000 -1.2644279e-002
6.9400000e+000 -4.2827397e-003
6.9600000e+000 1.6213229e-002
6.9800000e+000 4.8945600e-003
7.0000000e+000 -2.2331429e-002
7.0200000e+000 -4.7619989e-002
7.0400000e+000 -4.3643158e-002
7.0600000e+000 -2.2025520e-002
7.0800000e+000 -4.3847100e-003
7.1000000e+000 1.6213229e-002
7.1200000e+000 3.2630399e-002
7.1400000e+000 4.2725430e-002
7.1600000e+000 1.2542310e-002
7.1800000e+000 -1.6315199e-002
7.2000000e+000 -2.0801879e-002
7.2200000e+000 -8.3615395e-003
7.2400000e+000 -2.1005820e-002
7.2600000e+000 -1.3969889e-002
7.2800000e+000 -5.6083498e-003
7.3000000e+000 5.4044100e-003
7.3200000e+000 1.3663979e-002
7.3400000e+000 2.7124019e-002
7.3600000e+000 2.3657040e-002
7.3800000e+000 8.0556298e-003
7.4000000e+000 -8.1575998e-004
7.4200000e+000 2.0393999e-002
7.4400000e+000 4.4356948e-002
7.4600000e+000 5.0169239e-002
7.4800000e+000 1.9476270e-002
7.5000000e+000 9.3812395e-003
7.5200000e+000 -2.2433400e-003
7.5400000e+000 -2.1413698e-003
7.5600000e+000 5.3024396e-003
7.5800000e+000 9.4832099e-003
7.6000000e+000 2.6002349e-002
7.6200000e+000 3.7524958e-002
7.6400000e+000 5.3534248e-002
7.6600000e+000 5.5165767e-002
7.6800000e+000 4.3337248e-002
7.7000000e+000 4.0584058e-002
7.7200000e+000 5.7001230e-002
7.7400000e+000 7.7089316e-002
7.7600000e+000 3.7219049e-002
7.7800000e+000 4.1909667e-002
7.8000000e+000 9.9930598e-003
7.8200000e+000 -2.0801879e-002
7.8400000e+000 -2.5390529e-002
7.8600000e+000 -4.1297849e-002
7.8800000e+000 -4.2113608e-002
7.9000000e+000 -4.8027867e-002
7.9200000e+000 -4.4153008e-002
7.9400000e+000 -4.6702258e-002
7.9600000e+000 -5.8122896e-003
7.9800000e+000 1.8150659e-002
8.0000000e+000 -2.1209758e-002
8.0200000e+000 -5.0169239e-002
8.0400000e+000 -5.4044098e-002
8.0600000e+000 -3.6913139e-002
8.0800000e+000 -4.1297849e-002
8.1000000e+000 -3.1406758e-002
8.1200000e+000 -3.2222519e-002
8.1400000e+000 -2.7022049e-002
8.1600000e+000 -2.7022049e-002
8.1800000e+000 -2.7429929e-002
8.2000000e+000 -3.5179649e-002
8.2200000e+000 -3.1508729e-002
8.2400000e+000 -2.2127490e-002
8.2600000e+000 -7.9536599e-003
8.2800000e+000 8.8713895e-003
8.3000000e+000 2.8653569e-002
8.3200000e+000 3.1610699e-002
8.3400000e+000 3.6505258e-002
8.3600000e+000 3.4771767e-002
8.3800000e+000 3.6505258e-002
8.4000000e+000 2.9265390e-002
8.4200000e+000 3.1100849e-002
8.4400000e+000 1.1420639e-002
8.4600000e+000 2.1821579e-002
8.4800000e+000 1.3867920e-002
8.5000000e+000 3.9156480e-002
8.5200000e+000 -8.7796165e-002
8.5400000e+000 -1.3765950e-001
8.5600000e+000 -1.3663980e-001
8.5800000e+000 -1.3765950e-001
8.6000000e+000 -1.2134430e-001
8.6200000e+000 -1.0604880e-001
8.6400000e+000 -8.4533129e-002
8.6600000e+000 -6.6382466e-002
8.6800000e+000 -4.5274680e-002
8.7000000e+000 -2.6308258e-002
8.7200000e+000 -6.1181998e-003
8.7400000e+000 -9.2792701e-003
8.7600000e+000 -1.8558540e-002
8.7800000e+000 -1.4989589e-002
8.8000000e+000 8.6674497e-003
8.8200000e+000 1.6621109e-002
8.8400000e+000 5.0984998e-003
8.8600000e+000 2.6920079e-002
8.8800000e+000 5.9346539e-002
8.9000000e+000 8.8407984e-002
8.9200000e+000 1.2236400e-001
8.9400000e+000 1.7334899e-001
8.9600000e+000 1.1318670e-001
8.9800000e+000 -1.1216700e-001
9.0000000e+000 -3.7321017e-002
9.0200000e+000 -4.5376648e-002
9.0400000e+000 -2.4064920e-002
9.0600000e+000 -9.7891197e-002
9.0800000e+000 -6.6892316e-002
9.1000000e+000 -6.0876089e-002
9.1200000e+000 -6.8319898e-002
9.1400000e+000 -5.6287439e-002
9.1600000e+000 -2.7531900e-003
9.1800000e+000 3.8544658e-002
9.2000000e+000 1.0910790e-001
9.2200000e+000 1.7028989e-001
9.2400000e+000 9.6565584e-002
9.2600000e+000 4.1603758e-002
9.2800000e+000 6.8013985e-002
9.3000000e+000 1.3460039e-002
9.3200000e+000 -9.6871497e-003
9.3400000e+000 -5.3024398e-002
9.3600000e+000 -8.4329184e-002
9.3800000e+000 -1.1726550e-001
9.4000000e+000 -1.1726550e-001
9.4200000e+000 -8.1881910e-002
9.4400000e+000 -3.7626930e-002
9.4600000e+000 2.9571300e-003
9.4800000e+000 5.5573648e-002
9.5000000e+000 1.2032460e-001
9.5200000e+000 1.6417169e-001
9.5400000e+000 -2.7531899e-002
9.5600000e+000 3.4669800e-003
9.5800000e+000 -5.7103197e-003
9.6000000e+000 2.0393999e-003
9.6200000e+000 1.4887620e-002
9.6400000e+000 5.4757889e-002
9.6600000e+000 8.1372060e-002
9.6800000e+000 -2.0903849e-002
9.7000000e+000 -6.0162298e-002
9.7200000e+000 -1.7232929e-002
9.7400000e+000 -1.7844749e-002
9.7600000e+000 -2.8551599e-003
9.7800000e+000 7.5457798e-003
9.8000000e+000 3.8952539e-002
9.8200000e+000 5.7816989e-002
9.8400000e+000 7.6783410e-002
9.8600000e+000 8.1677966e-002
9.8800000e+000 6.0366239e-002
9.9000000e+000 3.0998879e-002
9.9200000e+000 2.3453099e-003
9.9400000e+000 6.5260799e-003
9.9600000e+000 -4.1399817e-002
9.9800000e+000 -4.5988467e-002
1.0000000e+001 -8.0556298e-003
1.0020000e+001 1.7130959e-002
1.0040000e+001 5.7816989e-002
1.0060000e+001 9.4832099e-003
1.0080000e+001 -5.6083498e-003
1.0100000e+001 4.4866799e-003
1.0120000e+001 -1.2542310e-002
1.0140000e+001 -2.8755540e-002
1.0160000e+001 -4.4560889e-002
1.0180000e+001 -3.5893440e-002
1.0200000e+001 -2.6002349e-002
1.0220000e+001 -1.1318670e-002
1.0240000e+001 2.0903849e-002
1.0260000e+001 5.2922430e-002
1.0280000e+001 8.7082379e-002
1.0300000e+001 1.1624580e-001
1.0320000e+001 7.4744011e-002
1.0340000e+001 2.4166890e-002
1.0360000e+001 -3.7524958e-002
1.0380000e+001 -2.7633869e-002
1.0400000e+001 -2.2127490e-002
1.0420000e+001 -8.9019810e-002
1.0440000e+001 -9.9216810e-002
1.0460000e+001 -6.0060329e-002
1.0480000e+001 -3.4261917e-002
1.0500000e+001 7.8516895e-003
1.0520000e+001 2.6410229e-002
1.0540000e+001 5.1800757e-002
1.0560000e+001 3.6811167e-002
1.0580000e+001 8.2595701e-003
1.0600000e+001 -5.7103197e-003
1.0620000e+001 -2.1311729e-002
1.0640000e+001 -3.2324490e-002
1.0660000e+001 -2.4268858e-002
1.0680000e+001 -3.8340717e-002
1.0700000e+001 -5.6083498e-002
1.0720000e+001 -7.3622334e-002
1.0740000e+001 -8.1881910e-002
1.0760000e+001 -5.3330307e-002
1.0780000e+001 -3.4669799e-002
1.0800000e+001 -1.1216700e-003
1.0820000e+001 6.6280498e-003
1.0840000e+001 -3.7728899e-003
1.0860000e+001 -5.0984998e-004
1.0880000e+001 -1.7130959e-002
1.0900000e+001 -4.1807699e-002
1.0920000e+001 -8.1575997e-003
1.0940000e+001 8.0556298e-003
1.0960000e+001 3.8136780e-002
1.0980000e+001 6.2711548e-002
1.1000000e+001 6.7810048e-002
1.1020000e+001 2.5900379e-002
1.1040000e+001 -5.8122896e-003
1.1060000e+001 -4.8333780e-002
1.1080000e+001 -3.6301317e-002
1.1100000e+001 -2.4778708e-002
1.1120000e+001 -4.8945600e-003
1.1140000e+001 1.2848220e-002
1.1160000e+001 3.8646630e-002
1.1180000e+001 2.4574770e-002
1.1200000e+001 -2.3147190e-002
1.1220000e+001 -4.3643158e-002
1.1240000e+001 -6.9237629e-002
1.1260000e+001 -6.7402166e-002
1.1280000e+001 -6.0162298e-002
1.1300000e+001 -5.2310607e-002
1.1320000e+001 -4.1603758e-002
1.1340000e+001 -3.1508729e-002
1.1360000e+001 -2.7124019e-002
1.1380000e+001 -5.5165767e-002
1.1400000e+001 -6.4037157e-002
1.1420000e+001 -9.2588760e-002
1.1440000e+001 -1.1318670e-001
1.1460000e+001 -8.9835565e-002
1.1480000e+001 -7.8516897e-002
1.1500000e+001 -5.9346539e-002
1.1520000e+001 -4.8231808e-002
1.1540000e+001 -3.3956008e-002
1.1560000e+001 -2.0292029e-002
1.1580000e+001 2.0393999e-003
1.1600000e+001 2.1515670e-002
1.1620000e+001 4.4051039e-002
1.1640000e+001 6.2507607e-002
1.1660000e+001 7.8210984e-002
1.1680000e+001 9.5138010e-002
1.1700000e+001 1.0910790e-001
1.1720000e+001 1.1522610e-001
1.1740000e+001 1.2134430e-001
1.1760000e+001 1.2746250e-001
1.1780000e+001 1.3562010e-001
1.1800000e+001 1.6213229e-001
1.1820000e+001 1.8354599e-001
1.1840000e+001 2.0801879e-001
1.1860000e+001 1.2644280e-001
1.1880000e+001 4.5070739e-002
1.1900000e+001 -1.4275800e-002
1.1920000e+001 -6.7912016e-002
1.1940000e+001 -5.6593348e-002
1.1960000e+001 -7.0665211e-002
1.1980000e+001 -1.0033848e-001
1.2000000e+001 -1.2746250e-001
1.2020000e+001 -1.2032460e-001
1.2040000e+001 -1.0706850e-001
1.2060000e+001 -9.3812397e-002
1.2080000e+001 -7.5763710e-002
1.2100000e+001 -8.2493729e-002
1.2120000e+001 -8.6674497e-002
1.2140000e+001 -8.7694197e-002
1.2160000e+001 -8.8000110e-002
1.2180000e+001 -8.9019810e-002
1.2200000e+001 -8.8509960e-002
1.2220000e+001 -9.0243447e-002
1.2240000e+001 -5.4757889e-002
1.2260000e+001 5.3024396e-003
1.2280000e+001 2.1923549e-002
1.2300000e+001 2.4982649e-002
1.2320000e+001 5.9142598e-002
1.2340000e+001 3.2018579e-002
1.2360000e+001 2.4064920e-002
1.2380000e+001 4.9455448e-002
1.2400000e+001 6.0060329e-002
1.2420000e+001 5.3534248e-002
1.2440000e+001 3.6199349e-002
1.2460000e+001 2.0088090e-002
1.2480000e+001 2.0292029e-002
1.2500000e+001 5.0169239e-002
1.2520000e+001 3.4975708e-002
1.2540000e+001 2.9367358e-002
1.2560000e+001 4.4051039e-002
1.2580000e+001 2.4370829e-002
1.2600000e+001 8.9733599e-003
1.2620000e+001 7.8516895e-003
1.2640000e+001 -1.5091560e-002
1.2660000e+001 -7.8516895e-003
1.2680000e+001 -1.9374299e-003
1.2700000e+001 7.6477497e-003
1.2720000e+001 4.4866799e-003
1.2740000e+001 -1.4785649e-002
1.2760000e+001 -3.2222519e-002
1.2780000e+001 -2.4574770e-002
1.2800000e+001 -2.8551599e-003
1.2820000e+001 1.8558540e-002
1.2840000e+001 4.3439217e-002
1.2860000e+001 4.4764830e-002
1.2880000e+001 5.2208639e-002
1.2900000e+001 4.7518017e-002
1.2920000e+001 4.8843630e-002
1.2940000e+001 1.9680209e-002
1.2960000e+001 2.2637340e-002
1.2980000e+001 2.7939779e-002
1.3000000e+001 4.0074208e-002
1.3020000e+001 5.1392880e-002
1.3040000e+001 5.8836689e-002
1.3060000e+001 5.9958357e-002
1.3080000e+001 8.3819334e-002
1.3100000e+001 8.1270084e-002
1.3120000e+001 9.6769528e-002
1.3140000e+001 3.5179649e-002
1.3160000e+001 4.5886498e-003
1.3180000e+001 -1.2542310e-002
1.3200000e+001 -3.5383590e-002
1.3220000e+001 -4.3439217e-002
1.3240000e+001 -4.2419517e-002
1.3260000e+001 -2.8041749e-002
1.3280000e+001 -2.7531899e-002
1.3300000e+001 7.5457798e-003
1.3320000e+001 4.3643158e-002
1.3340000e+001 -2.3555070e-002
1.3360000e+001 -3.9462389e-002
1.3380000e+001 -8.4635099e-003
1.3400000e+001 1.4173829e-002
1.3420000e+001 4.5376648e-002
1.3440000e+001 2.7531900e-003
1.3460000e+001 -7.1073084e-002
1.3480000e+001 -8.1168116e-002
1.3500000e+001 -2.5594470e-002
1.3520000e+001 -1.3765950e-002
1.3540000e+001 8.0556298e-003
1.3560000e+001 -1.1726550e-002
1.3580000e+001 -2.5594470e-002
1.3600000e+001 -3.3956008e-002
1.3620000e+001 -2.7429929e-002
1.3640000e+001 -3.0692969e-002
1.3660000e+001 -2.0393999e-002
1.3680000e+001 -6.8319896e-003
1.3700000e+001 -3.8748598e-003
1.3720000e+001 1.0706850e-002
1.3740000e+001 3.0183119e-002
1.3760000e+001 3.5077680e-002
1.3780000e+001 9.7585284e-002
1.3800000e+001 9.1569060e-002
1.3820000e+001 1.8252629e-002
1.3840000e+001 -3.6913139e-002
1.3860000e+001 -1.0135818e-001
1.3880000e+001 -8.2289784e-002
1.3900000e+001 -7.5865679e-002
1.3920000e+001 -5.4961830e-002
1.3940000e+001 -3.3650099e-002
1.3960000e+001 -1.3052160e-002
1.3980000e+001 3.1610698e-003
1.4000000e+001 1.5091560e-002
1.4020000e+001 5.1800757e-002
1.4040000e+001 -2.2433400e-003
1.4060000e+001 -4.9863330e-002
1.4080000e+001 -3.6505258e-002
1.4100000e+001 -7.0461266e-002
1.4120000e+001 -5.2616517e-002
1.4140000e+001 -3.7830867e-002
1.4160000e+001 8.9733599e-003
1.4180000e+001 6.4445039e-002
1.4200000e+001 8.5756765e-002
1.4220000e+001 1.3052160e-001
1.4240000e+001 1.4173830e-001
1.4260000e+001 1.2134430e-001
1.4280000e+001 7.6579466e-002
1.4300000e+001 2.2943249e-002
1.4320000e+001 -8.9733599e-003
1.4340000e+001 -2.3147190e-002
1.4360000e+001 7.5457798e-003
1.4380000e+001 1.8456570e-002
1.4400000e+001 5.5471680e-002
1.4420000e+001 4.0686030e-002
1.4440000e+001 4.5886498e-003
1.4460000e+001 -8.3615395e-003
1.4480000e+001 -1.8864449e-002
1.4500000e+001 -2.0393999e-003
1.4520000e+001 6.1182000e-004
1.4540000e+001 -1.1930489e-002
1.4560000e+001 -2.1413699e-002
1.4580000e+001 -3.0896908e-002
1.4600000e+001 -5.2208639e-002
1.4620000e+001 -7.4132184e-002
1.4640000e+001 -5.9040630e-002
1.4660000e+001 -2.7124019e-002
1.4680000e+001 -1.8150659e-002
1.4700000e+001 4.0787999e-003
1.4720000e+001 9.9930598e-003
1.4740000e+001 1.3969889e-002
1.4760000e+001 2.2535370e-002
1.4780000e+001 4.4560889e-002
1.4800000e+001 9.2792701e-003
1.4820000e+001 -5.5879557e-002
1.4840000e+001 -5.6593348e-002
1.4860000e+001 -2.4778708e-002
1.4880000e+001 -8.2595701e-003
1.4900000e+001 2.5492499e-002
1.4920000e+001 4.1807699e-002
1.4940000e+001 1.8558540e-002
1.4960000e+001 -2.7531900e-003
1.4980000e+001 -2.4778708e-002
1.5000000e+001 -1.5295499e-003
1.5020000e+001 2.5186590e-002
1.5040000e+001 4.9149539e-002
1.5060000e+001 7.9842510e-002
1.5080000e+001 6.3425339e-002
1.5100000e+001 3.3752067e-002
1.5120000e+001 -1.4275799e-003
1.5140000e+001 -1.9884149e-002
1.5160000e+001 -2.5186590e-002
1.5180000e+001 -2.1617640e-002
1.5200000e+001 -1.1216700e-002
1.5220000e+001 5.0984998e-003
1.5240000e+001 2.4574770e-002
1.5260000e+001 -3.4669800e-003
1.5280000e+001 -2.2025520e-002
1.5300000e+001 -4.8027867e-002
1.5320000e+001 -3.7015108e-002
1.5340000e+001 -1.9884149e-002
1.5360000e+001 -1.8354599e-003
1.5380000e+001 1.7334899e-002
1.5400000e+001 -8.1575997e-003
1.5420000e+001 5.0984998e-004
1.5440000e+001 2.3453099e-002
1.5460000e+001 3.8136780e-002
1.5480000e+001 6.1283966e-002
1.5500000e+001 5.2616517e-002
1.5520000e+001 4.4051039e-002
1.5540000e+001 3.5077680e-002
1.5560000e+001 5.1494848e-002
1.5580000e+001 6.6586411e-002
1.5600000e+001 6.9645511e-002
1.5620000e+001 1.7538840e-002
1.5640000e+001 -1.7334899e-002
1.5660000e+001 -5.3738189e-002
1.5680000e+001 -6.7708079e-002
1.5700000e+001 -3.9462389e-002
1.5720000e+001 -2.2637340e-002
1.5740000e+001 -3.3650098e-003
1.5760000e+001 1.2134429e-002
1.5780000e+001 -1.3052160e-002
1.5800000e+001 -3.5791467e-002
1.5820000e+001 -5.2412580e-002
1.5840000e+001 -3.4159949e-002
1.5860000e+001 -2.2229458e-002
1.5880000e+001 -1.2236400e-003
1.5900000e+001 1.4479739e-002
1.5920000e+001 7.1378998e-003
1.5940000e+001 -6.4241100e-003
1.5960000e+001 -1.2236400e-002
1.5980000e+001 -3.2834340e-002
1.6000000e+001 -3.5281617e-002
1.6020000e+001 -9.2792701e-003
1.6040000e+001 7.4438099e-003
1.6060000e+001 3.1508729e-002
1.6080000e+001 4.8129839e-002
1.6100000e+001 6.1487907e-002
1.6120000e+001 5.8734716e-002
1.6140000e+001 3.3650099e-002
1.6160000e+001 -7.4438099e-003
1.6180000e+001 -7.9230684e-002
1.6200000e+001 -6.1997757e-002
1.6220000e+001 -4.4662858e-002
1.6240000e+001 -2.1311729e-002
1.6260000e+001 3.1610698e-003
1.6280000e+001 3.5689499e-002
1.6300000e+001 2.9877208e-002
1.6320000e+001 1.2338370e-002
1.6340000e+001 3.4465858e-002
1.6360000e+001 3.2324490e-002
1.6380000e+001 2.5900379e-002
1.6400000e+001 2.1005820e-002
1.6420000e+001 2.0190059e-002
1.6440000e+001 1.7742779e-002
1.6460000e+001 2.1413698e-003
1.6480000e+001 -1.4683679e-002
1.6500000e+001 -3.4975708e-002
1.6520000e+001 -3.4567830e-002
1.6540000e+001 -1.4785649e-002
1.6560000e+001 -2.8551599e-003
1.6580000e+001 1.7334899e-002
1.6600000e+001 -9.7891200e-003
1.6620000e+001 -2.6002349e-002
1.6640000e+001 -2.8449629e-002
1.6660000e+001 -3.9564358e-002
1.6680000e+001 -2.4676740e-002
1.6700000e+001 -2.1923549e-002
1.6720000e+001 -1.8558540e-002
1.6740000e+001 -1.7742779e-002
1.6760000e+001 -3.8748598e-003
1.6780000e+001 -2.7531900e-003
1.6800000e+001 -1.8864449e-002
1.6820000e+001 -1.2542310e-002
1.6840000e+001 8.8713895e-003
1.6860000e+001 3.4975708e-002
1.6880000e+001 7.0869148e-002
1.6900000e+001 9.2792697e-002
1.6920000e+001 8.6980410e-002
1.6940000e+001 7.7497197e-002
1.6960000e+001 5.2310607e-002
1.6980000e+001 1.8966420e-002
1.7000000e+001 1.5295499e-003
1.7020000e+001 -1.9374299e-002
1.7040000e+001 -1.5397470e-002
1.7060000e+001 -7.4438099e-003
1.7080000e+001 2.1413698e-003
1.7100000e+001 1.3154129e-002
1.7120000e+001 2.1923549e-002
1.7140000e+001 2.4472800e-003
1.7160000e+001 -1.2644279e-002
1.7180000e+001 -3.3548130e-002
1.7200000e+001 -5.2922430e-002
1.7220000e+001 -7.2194761e-002
1.7240000e+001 -5.9040630e-002
1.7260000e+001 -4.7110139e-002
1.7280000e+001 -3.1304790e-002
1.7300000e+001 -1.4785649e-002
1.7320000e+001 -9.1772994e-004
1.7340000e+001 -1.8354599e-002
1.7360000e+001 -3.2426458e-002
1.7380000e+001 -4.7416048e-002
1.7400000e+001 -3.9870267e-002
1.7420000e+001 -3.5179649e-002
1.7440000e+001 -3.2222519e-002
1.7460000e+001 -4.4356948e-002
1.7480000e+001 -5.0067267e-002
1.7500000e+001 -4.8435748e-002
1.7520000e+001 -4.2827399e-002
1.7540000e+001 -3.6811167e-002
1.7560000e+001 -2.8245690e-002
1.7580000e+001 -2.6308258e-002
1.7600000e+001 -1.4173829e-002
1.7620000e+001 -6.9339600e-003
1.7640000e+001 5.1698789e-002
1.7660000e+001 7.3622334e-002
1.7680000e+001 8.9529660e-002
1.7700000e+001 7.9740534e-002
1.7720000e+001 7.8007047e-002
1.7740000e+001 4.4764830e-002
1.7760000e+001 8.1575997e-003
1.7780000e+001 1.3256099e-003
1.7800000e+001 -1.2848220e-002
1.7820000e+001 -1.5295499e-003
1.7840000e+001 3.0590999e-003
1.7860000e+001 1.0604879e-002
1.7880000e+001 1.0604879e-002
1.7900000e+001 1.9680209e-002
1.7920000e+001 2.0903849e-002
1.7940000e+001 7.5457798e-003
1.7960000e+001 -5.7103197e-003
1.7980000e+001 -7.3418396e-003
1.8000000e+001 7.1378998e-003
1.8020000e+001 1.0808820e-002
1.8040000e+001 1.4989589e-002
1.8060000e+001 -9.1772994e-004
1.8080000e+001 -1.6213229e-002
1.8100000e+001 -1.9068390e-002
1.8120000e+001 -7.1378996e-004
1.8140000e+001 1.5805349e-002
1.8160000e+001 1.0706850e-002
1.8180000e+001 -1.1726550e-002
1.8200000e+001 -3.0794940e-002
1.8220000e+001 -3.1508729e-002
1.8240000e+001 -9.6871497e-003
1.8260000e+001 -5.9142600e-003
1.8280000e+001 4.0787999e-004
1.8300000e+001 2.0393999e-003
1.8320000e+001 5.0984998e-003
1.8340000e+001 5.8122896e-003
1.8360000e+001 9.8910895e-003
1.8380000e+001 1.3663979e-002
1.8400000e+001 1.8048690e-002
1.8420000e+001 2.2229458e-002
1.8440000e+001 2.6614169e-002
1.8460000e+001 3.0794940e-002
1.8480000e+001 3.5281617e-002
1.8500000e+001 3.9360417e-002
1.8520000e+001 4.8333780e-002
1.8540000e+001 4.0074208e-002
1.8560000e+001 2.4268858e-002
1.8580000e+001 1.1726550e-002
1.8600000e+001 -8.0556298e-003
1.8620000e+001 -1.2644279e-002
1.8640000e+001 5.5063799e-003
1.8660000e+001 2.7531900e-003
1.8680000e+001 -2.5492499e-002
1.8700000e+001 -5.7715016e-002
1.8720000e+001 -6.4241098e-002
1.8740000e+001 -6.0264266e-002
1.8760000e+001 -4.2113608e-002
1.8780000e+001 -6.9339600e-003
1.8800000e+001 2.7735840e-002
1.8820000e+001 2.8245690e-002
1.8840000e+001 -2.1413698e-003
1.8860000e+001 -6.1181998e-003
1.8880000e+001 -1.1216700e-002
1.8900000e+001 -2.2535370e-002
1.8920000e+001 -4.2419517e-002
1.8940000e+001 -5.2922430e-002
1.8960000e+001 -2.2637340e-002
1.8980000e+001 3.0590999e-003
1.9000000e+001 8.0556298e-003
1.9020000e+001 1.4173829e-002
1.9040000e+001 1.7436870e-002
1.9060000e+001 2.5798408e-002
1.9080000e+001 3.2936308e-002
1.9100000e+001 3.9870267e-002
1.9120000e+001 1.6723079e-002
1.9140000e+001 -1.3867920e-002
1.9160000e+001 -3.2936308e-002
1.9180000e+001 -2.9673269e-002
1.9200000e+001 -2.9265390e-002
1.9220000e+001 -3.0998879e-002
1.9240000e+001 -3.4567830e-002
1.9260000e+001 -2.4982649e-002
1.9280000e+001 -7.7497196e-003
1.9300000e+001 1.2746250e-002
1.9320000e+001 3.8340717e-002
1.9340000e+001 4.0991939e-002
1.9360000e+001 2.4982649e-002
1.9380000e+001 1.5907320e-002
1.9400000e+001 -4.0787999e-003
1.9420000e+001 -1.5601410e-002
1.9440000e+001 -2.9469329e-002
1.9460000e+001 -3.2222519e-002
1.9480000e+001 -1.1318670e-002
1.9500000e+001 9.5851793e-003
1.9520000e+001 3.4159949e-002
1.9540000e+001 5.8734716e-002
1.9560000e+001 4.3235280e-002
1.9580000e+001 1.4581710e-002
1.9600000e+001 -7.1378996e-004
1.9620000e+001 -1.3765950e-002
1.9640000e+001 -2.7531899e-002
1.9660000e+001 -3.4771767e-002
1.9680000e+001 -3.6403290e-002
1.9700000e+001 -4.0380117e-002
1.9720000e+001 -4.0991939e-002
1.9740000e+001 -4.9761357e-002
1.9760000e+001 -4.8945598e-002
1.9780000e+001 -4.1399817e-002
1.9800000e+001 -4.1501789e-002
1.9820000e+001 -3.5791467e-002
1.9840000e+001 -1.9068390e-002
1.9860000e+001 -5.8122896e-003
1.9880000e+001 4.4866799e-003
1.9900000e+001 -1.9374299e-003
1.9920000e+001 -7.3418396e-003
1.9940000e+001 -1.7232929e-002
1.9960000e+001 -1.1726550e-002
1.9980000e+001 1.2848220e-002
2.0000000e+001 3.6505258e-002
2.0020000e+001 6.6688379e-002
2.0040000e+001 7.3010516e-002
2.0060000e+001 7.7701134e-002
2.0080000e+001 7.5355829e-002
2.0100000e+001 6.4037157e-002
2.0120000e+001 4.9353480e-002
2.0140000e+001 2.6920079e-002
2.0160000e+001 -4.4866799e-003
2.0180000e+001 -2.9367358e-002
2.0200000e+001 -3.9156480e-002
2.0220000e+001 -5.0169239e-002
2.0240000e+001 -4.3643158e-002
2.0260000e+001 -4.2419517e-002
2.0280000e+001 -2.8143719e-002
2.0300000e+001 -5.3024396e-003
2.0320000e+001 2.4166890e-002
2.0340000e+001 4.3439217e-002
2.0360000e+001 6.1589879e-002
2.0380000e+001 4.6090439e-002
2.0400000e+001 2.8959479e-002
2.0420000e+001 1.2848220e-002
2.0440000e+001 -5.5063799e-003
2.0460000e+001 -2.8041749e-002
2.0480000e+001 -4.3133308e-002
2.0500000e+001 -1.7844749e-002
2.0520000e+001 1.0197000e-004
2.0540000e+001 2.5900379e-002
2.0560000e+001 4.9047567e-002
2.0580000e+001 6.5362766e-002
2.0600000e+001 5.6491380e-002
2.0620000e+001 4.2113608e-002
2.0640000e+001 1.8762479e-002
2.0660000e+001 -4.8945600e-003
2.0680000e+001 -3.0896908e-002
2.0700000e+001 -5.4146067e-002
2.0720000e+001 -7.2194761e-002
2.0740000e+001 -9.4628160e-002
2.0760000e+001 -8.8000110e-002
2.0780000e+001 -6.4343066e-002
2.0800000e+001 -3.8340717e-002
2.0820000e+001 8.8713895e-003
2.0840000e+001 3.1508729e-002
2.0860000e+001 6.0060329e-002
2.0880000e+001 6.2609579e-002
2.0900000e+001 3.9258449e-002
2.0920000e+001 3.5791467e-002
2.0940000e+001 3.1712669e-002
2.0960000e+001 2.5084620e-002
2.0980000e+001 1.9374299e-003
2.1000000e+001 -2.0190059e-002
2.1020000e+001 -1.6111260e-002
2.1040000e+001 -1.5295499e-003
2.1060000e+001 1.0095029e-002
2.1080000e+001 2.9163419e-002
2.1100000e+001 4.1603758e-002
2.1120000e+001 5.7409107e-002
2.1140000e+001 5.4146067e-002
2.1160000e+001 3.2018579e-002
2.1180000e+001 1.6825049e-002
2.1200000e+001 -2.4472800e-003
2.1220000e+001 -1.9272329e-002
2.1240000e+001 -2.8143719e-002
2.1260000e+001 -3.7830867e-002
2.1280000e+001 -4.5886498e-002
2.1300000e+001 -5.4451980e-002
2.1320000e+001 -4.9251508e-002
2.1340000e+001 -3.8646630e-002
2.1360000e+001 -3.0183119e-002
2.1380000e+001 -1.9986120e-002
2.1400000e+001 -1.8762479e-002
2.1420000e+001 -1.6213229e-002
2.1440000e+001 -1.2134429e-002
2.1460000e+001 -5.4044100e-003
2.1480000e+001 2.0394000e-004
2.1500000e+001 6.0162299e-003
2.1520000e+001 -2.3453099e-003
2.1540000e+001 -1.1420639e-002
2.1560000e+001 -2.0903849e-002
2.1580000e+001 -3.2834340e-002
2.1600000e+001 -3.9564358e-002
2.1620000e+001 -3.4771767e-002
2.1640000e+001 -2.9265390e-002
2.1660000e+001 -3.3446158e-002
2.1680000e+001 -4.1501789e-002
2.1700000e+001 -4.9659389e-002
2.1720000e+001 -5.7409107e-002
2.1740000e+001 -6.5668679e-002
2.1760000e+001 -5.6593348e-002
2.1780000e+001 -4.5478617e-002
2.1800000e+001 -8.1575998e-004
2.1820000e+001 2.5798408e-002
2.1840000e+001 4.1909667e-002
2.1860000e+001 6.5668679e-002
2.1880000e+001 5.9040630e-002
2.1900000e+001 4.8333780e-002
2.1920000e+001 3.9156480e-002
2.1940000e+001 3.9258449e-002
2.1960000e+001 3.4669799e-002
2.1980000e+001 3.6403290e-002
2.2000000e+001 8.1575998e-004
2.2020000e+001 -2.5900379e-002
2.2040000e+001 -4.6906198e-002
2.2060000e+001 -4.8027867e-002
2.2080000e+001 -2.2637340e-002
2.2100000e+001 -6.6280498e-003
2.2120000e+001 1.6723079e-002
2.2140000e+001 3.6199349e-002
2.2160000e+001 5.1392880e-002
2.2180000e+001 3.7830867e-002
2.2200000e+001 2.8551599e-002
2.2220000e+001 1.6111260e-002
2.2240000e+001 4.1807698e-003
2.2260000e+001 -2.7531900e-003
2.2280000e+001 2.1413698e-003
2.2300000e+001 4.4866799e-003
2.2320000e+001 1.0095029e-002
2.2340000e+001 1.3765950e-002
2.2360000e+001 9.5851793e-003
2.2380000e+001 5.9142600e-003
2.2400000e+001 2.2433400e-003
2.2420000e+001 -3.1610698e-003
2.2440000e+001 -3.1610698e-003
2.2460000e+001 4.0787999e-004
2.2480000e+001 -2.5492499e-003
2.2500000e+001 -1.2644279e-002
2.2520000e+001 -2.3962949e-002
2.2540000e+001 -4.1399817e-002
2.2560000e+001 -5.4044098e-002
2.2580000e+001 -7.1480966e-002
2.2600000e+001 -3.2936308e-002
2.2620000e+001 -5.3024396e-003
2.2640000e+001 9.5851793e-003
2.2660000e+001 3.3446158e-002
2.2680000e+001 4.8741658e-002
2.2700000e+001 5.1902730e-002
2.2720000e+001 3.6505258e-002
2.2740000e+001 3.4873740e-002
2.2760000e+001 3.1202819e-002
2.2780000e+001 2.9061449e-002
2.2800000e+001 2.6818108e-002
2.2820000e+001 1.7130959e-002
2.2840000e+001 -5.0984998e-004
2.2860000e+001 -2.1617640e-002
2.2880000e+001 -3.7219049e-002
2.2900000e+001 -3.1610699e-002
2.2920000e+001 -3.0285090e-002
2.2940000e+001 -2.8551599e-002
2.2960000e+001 -2.4166890e-002
2.2980000e+001 -2.7124019e-002
2.3000000e+001 -3.1406758e-002
2.3020000e+001 -3.7321017e-002
2.3040000e+001 -3.6403290e-002
2.3060000e+001 -3.1406758e-002
2.3080000e+001 -1.9680209e-002
2.3100000e+001 1.9374299e-003
2.3120000e+001 1.9986120e-002
2.3140000e+001 1.6315199e-002
2.3160000e+001 1.3154129e-002
2.3180000e+001 1.4275800e-002
2.3200000e+001 1.1216700e-002
2.3220000e+001 1.1012760e-002
2.3240000e+001 9.3812395e-003
2.3260000e+001 9.0753293e-003
2.3280000e+001 1.9374299e-003
2.3300000e+001 -1.3358070e-002
2.3320000e+001 -2.5186590e-002
2.3340000e+001 -4.4458917e-002
2.3360000e+001 -4.4051039e-002
2.3380000e+001 -3.0590999e-002
2.3400000e+001 -1.9578240e-002
2.3420000e+001 -4.8945600e-003
2.3440000e+001 9.8910895e-003
2.3460000e+001 1.7130959e-002
2.3480000e+001 1.5091560e-002
2.3500000e+001 1.7640809e-002
2.3520000e+001 7.9536599e-003
2.3540000e+001 -5.9142600e-003
2.3560000e+001 -2.1923549e-002
2.3580000e+001 -2.3860979e-002
2.3600000e+001 6.1181998e-003
2.3620000e+001 2.6716140e-002
2.3640000e+001 2.7429929e-002
2.3660000e+001 8.5654793e-003
2.3680000e+001 -4.1807698e-003
2.3700000e+001 -2.3147190e-002
2.3720000e+001 -7.7497196e-003
2.3740000e+001 3.1610698e-003
2.3760000e+001 1.8558540e-002
2.3780000e+001 3.1610699e-002
2.3800000e+001 4.8843630e-002
2.3820000e+001 4.6804230e-002
2.3840000e+001 1.6927020e-002
2.3860000e+001 -5.9142600e-003
2.3880000e+001 -4.0380117e-002
2.3900000e+001 -4.5274680e-002
2.3920000e+001 -2.4574770e-002
2.3940000e+001 -1.0298970e-002
2.3960000e+001 2.2127490e-002
2.3980000e+001 2.6614169e-002
2.4000000e+001 1.2338370e-002
2.4020000e+001 8.1575998e-004
2.4040000e+001 -1.7130959e-002
2.4060000e+001 -3.8952539e-002
2.4080000e+001 -5.7715016e-002
2.4100000e+001 -7.9638566e-002
2.4120000e+001 -6.3119429e-002
2.4140000e+001 -2.6104319e-002
2.4160000e+001 4.4866799e-003
2.4180000e+001 4.6498317e-002
2.4200000e+001 7.9842510e-002
2.4220000e+001 1.1216700e-001
2.4240000e+001 9.7177410e-002
2.4260000e+001 4.9863330e-002
2.4280000e+001 1.2440339e-002
2.4300000e+001 -3.9462389e-002
2.4320000e+001 -8.6266615e-002
2.4340000e+001 -1.2542310e-001
2.4360000e+001 -8.8102079e-002
2.4380000e+001 -6.2813516e-002
2.4400000e+001 -2.3249158e-002
2.4420000e+001 1.5295499e-002
2.4440000e+001 4.8639689e-002
2.4460000e+001 3.2936308e-002
2.4480000e+001 2.5288558e-002
2.4500000e+001 9.6871497e-003
2.4520000e+001 2.3147190e-002
2.4540000e+001 3.4465858e-002
2.4560000e+001 5.1494848e-002
2.4580000e+001 6.0570179e-002
2.4600000e+001 5.2922430e-002
2.4620000e+001 5.6287439e-002
2.4640000e+001 6.0672148e-002
2.4660000e+001 6.2915489e-002
2.4680000e+001 6.0468207e-002
2.4700000e+001 4.7518017e-002
2.4720000e+001 1.6111260e-002
2.4740000e+001 -7.8516895e-003
2.4760000e+001 -2.3453099e-002
2.4780000e+001 -3.0896908e-002
2.4800000e+001 -3.3548130e-002
2.4820000e+001 -3.7117080e-002
2.4840000e+001 -4.9965298e-002
2.4860000e+001 -6.7300198e-002
2.4880000e+001 -7.3724311e-002
2.4900000e+001 -7.9944479e-002
2.4920000e+001 -8.6266615e-002
2.4940000e+001 -5.7715016e-002
2.4960000e+001 -2.1311729e-002
2.4980000e+001 1.3052160e-002
2.5000000e+001 3.8850567e-002
2.5020000e+001 4.9761357e-002
2.5040000e+001 3.7015108e-002
2.5060000e+001 2.5492499e-002
2.5080000e+001 1.6111260e-002
2.5100000e+001 3.8442689e-002
2.5120000e+001 6.0570179e-002
2.5140000e+001 6.4241098e-002
2.5160000e+001 3.7728899e-002
2.5180000e+001 2.2127490e-002
2.5200000e+001 4.0787999e-003
2.5220000e+001 2.3453099e-003
2.5240000e+001 -6.3221396e-003
2.5260000e+001 -2.9163419e-002
2.5280000e+001 -3.8748599e-002
2.5300000e+001 -4.6600289e-002
2.5320000e+001 -5.1494848e-002
2.5340000e+001 -7.9230684e-002
2.5360000e+001 -6.4037157e-002
2.5380000e+001 -1.9272329e-002
2.5400000e+001 1.9272329e-002
2.5420000e+001 5.1902730e-002
2.5440000e+001 7.5865679e-002
2.5460000e+001 7.6375529e-002
2.5480000e+001 5.7307139e-002
2.5500000e+001 4.6396348e-002
2.5520000e+001 5.3534248e-002
2.5540000e+001 6.4750948e-002
2.5560000e+001 7.4540066e-002
2.5580000e+001 8.6776465e-002
2.5600000e+001 9.4526184e-002
2.5620000e+001 9.9420747e-002
2.5640000e+001 9.4730128e-002
2.5660000e+001 8.3003579e-002
2.5680000e+001 2.6920079e-002
2.5700000e+001 -2.9469329e-002
2.5720000e+001 -9.1874965e-002
2.5740000e+001 -1.3867920e-001
2.5760000e+001 -8.7592229e-002
2.5780000e+001 -4.7721958e-002
2.5800000e+001 8.2595701e-003
2.5820000e+001 6.0570179e-002
2.5840000e+001 8.3921310e-002
2.5860000e+001 5.6695316e-002
2.5880000e+001 3.0692969e-002
2.5900000e+001 -1.1216700e-003
2.5920000e+001 -3.8748599e-002
2.5940000e+001 -3.5383590e-002
2.5960000e+001 -2.0393999e-003
2.5980000e+001 1.8354599e-002
2.6000000e+001 6.4547007e-002
2.6020000e+001 1.0298970e-001
2.6040000e+001 1.1420640e-001
2.6060000e+001 1.1318670e-001
2.6080000e+001 9.5851797e-002
2.6100000e+001 8.2085847e-002
2.6120000e+001 6.1997757e-002
2.6140000e+001 5.1800757e-002
2.6160000e+001 2.1005820e-002
2.6180000e+001 -2.0495970e-002
2.6200000e+001 -5.8326839e-002
2.6220000e+001 -1.0400940e-001
2.6240000e+001 -1.3562010e-001
2.6260000e+001 -1.2542310e-001
2.6280000e+001 -1.2338370e-001
2.6300000e+001 -1.0298970e-001
2.6320000e+001 -7.7701134e-002
2.6340000e+001 -5.5879557e-002
2.6360000e+001 -4.9863330e-002
2.6380000e+001 -4.1297849e-002
2.6400000e+001 -2.9673269e-002
2.6420000e+001 -1.7946720e-002
2.6440000e+001 -1.4377770e-002
2.6460000e+001 -9.9930598e-003
2.6480000e+001 -5.0984998e-003
2.6500000e+001 1.9374299e-003
2.6520000e+001 8.1575997e-003
2.6540000e+001 3.5689499e-003
2.6560000e+001 -2.9571300e-003
2.6580000e+001 -6.2201697e-003
2.6600000e+001 -3.4669800e-003
2.6620000e+001 -1.3256099e-003
2.6640000e+001 7.7497196e-003
2.6660000e+001 2.0903849e-002
2.6680000e+001 3.3344190e-002
2.6700000e+001 4.5886498e-002
2.6720000e+001 5.8836689e-002
2.6740000e+001 5.7307139e-002
2.6760000e+001 4.7314080e-002
2.6780000e+001 3.0590999e-002
2.6800000e+001 1.0910789e-002
2.6820000e+001 -4.0787999e-004
2.6840000e+001 -3.3650098e-003
2.6860000e+001 -7.0359299e-003
2.6880000e+001 -1.0298970e-002
2.6900000e+001 -2.0495970e-002
2.6920000e+001 -1.9170359e-002
2.6940000e+001 2.5492499e-003
2.6960000e+001 1.3663979e-002
2.6980000e+001 2.4778708e-002
2.7000000e+001 2.7124019e-002
2.7020000e+001 2.7022049e-002
2.7040000e+001 1.5805349e-002
2.7060000e+001 -1.9374299e-003
2.7080000e+001 -9.6871497e-003
2.7100000e+001 -2.2025520e-002
2.7120000e+001 -1.7028990e-002
2.7140000e+001 -1.2236400e-002
2.7160000e+001 -6.5260799e-003
2.7180000e+001 -1.3154129e-002
2.7200000e+001 -1.6621109e-002
2.7220000e+001 -1.9680209e-002
2.7240000e+001 -2.4676740e-002
2.7260000e+001 -2.4064920e-002
2.7280000e+001 -1.7844749e-002
2.7300000e+001 -1.2644279e-002
2.7320000e+001 -1.8864449e-002
2.7340000e+001 -2.7022049e-002
2.7360000e+001 -3.2936308e-002
2.7380000e+001 -3.4261917e-002
2.7400000e+001 -4.6294380e-002
2.7420000e+001 -4.3847098e-002
2.7440000e+001 -3.4057980e-002
2.7460000e+001 -2.1719608e-002
2.7480000e+001 -7.0359299e-003
2.7500000e+001 3.0590999e-003
2.7520000e+001 3.0591000e-004
2.7540000e+001 -9.4832099e-003
2.7560000e+001 -9.0753293e-003
2.7580000e+001 -1.5295499e-002
2.7600000e+001 -1.6723079e-002
2.7620000e+001 -2.4268858e-002
2.7640000e+001 -3.2936308e-002
2.7660000e+001 -4.2929367e-002
2.7680000e+001 -4.6600289e-002
2.7700000e+001 -4.0482089e-002
2.7720000e+001 -3.5587530e-002
2.7740000e+001 -2.6308258e-002
2.7760000e+001 -1.7538840e-002
2.7780000e+001 -2.0393999e-003
2.7800000e+001 1.5907320e-002
2.7820000e+001 2.8959479e-002
2.7840000e+001 3.6913139e-002
2.7860000e+001 3.6097380e-002
2.7880000e+001 2.7429929e-002
2.7900000e+001 1.0298970e-002
2.7920000e+001 -4.5886498e-003
2.7940000e+001 -1.2746250e-002
2.7960000e+001 -2.4982649e-002
2.7980000e+001 -2.3351129e-002
2.8000000e+001 -1.2848220e-002
2.8020000e+001 -6.9339600e-003
2.8040000e+001 1.8354599e-003
2.8060000e+001 9.4832099e-003
2.8080000e+001 2.0393999e-002
2.8100000e+001 2.9163419e-002
2.8120000e+001 3.7219049e-002
2.8140000e+001 3.1712669e-002
2.8160000e+001 1.8456570e-002
2.8180000e+001 2.4472800e-003
2.8200000e+001 -1.5907320e-002
2.8220000e+001 -3.2528428e-002
2.8240000e+001 -2.2331429e-002
2.8260000e+001 -1.2032460e-002
2.8280000e+001 1.5295499e-003
2.8300000e+001 1.5601410e-002
2.8320000e+001 3.0387058e-002
2.8340000e+001 2.4778708e-002
2.8360000e+001 1.3867920e-002
2.8380000e+001 1.0400939e-002
2.8400000e+001 2.4472800e-003
2.8420000e+001 -1.0197000e-003
2.8440000e+001 -2.3453099e-003
2.8460000e+001 -3.6709198e-003
2.8480000e+001 -4.8945600e-003
2.8500000e+001 -9.0753293e-003
2.8520000e+001 -1.5805349e-002
2.8540000e+001 -1.5091560e-002
2.8560000e+001 -7.7497196e-003
2.8580000e+001 0.0000000e+000
2.8600000e+001 -2.0393999e-003
2.8620000e+001 -1.5091560e-002
2.8640000e+001 -2.2943249e-002
2.8660000e+001 -3.8136780e-002
2.8680000e+001 -3.7219049e-002
2.8700000e+001 -2.5594470e-002
2.8720000e+001 -1.6723079e-002
2.8740000e+001 -1.4275799e-003
2.8760000e+001 1.5601410e-002
2.8780000e+001 3.0794940e-002
2.8800000e+001 4.0074208e-002
2.8820000e+001 4.1399817e-002
2.8840000e+001 3.9258449e-002
2.8860000e+001 3.3548130e-002
2.8880000e+001 1.8660509e-002
2.8900000e+001 1.2746250e-002
2.8920000e+001 7.1378998e-003
2.8940000e+001 -1.0197000e-004
2.8960000e+001 -6.9339600e-003
2.8980000e+001 -1.3256100e-002
2.9000000e+001 -1.3052160e-002
2.9020000e+001 -1.1726550e-002
2.9040000e+001 -1.0400939e-002
2.9060000e+001 -8.5654793e-003
2.9080000e+001 -1.4377770e-002
2.9100000e+001 -2.0903849e-002
2.9120000e+001 -2.7429929e-002
2.9140000e+001 -3.5689499e-002
2.9160000e+001 -3.6607230e-002
2.9180000e+001 -3.0285090e-002
2.9200000e+001 -2.3147190e-002
2.9220000e+001 1.8354599e-003
2.9240000e+001 1.0095029e-002
2.9260000e+001 2.0699909e-002
2.9280000e+001 2.6104319e-002
2.9300000e+001 2.0292029e-002
2.9320000e+001 1.6315199e-002
2.9340000e+001 1.0910789e-002
2.9360000e+001 1.5907320e-002
2.9380000e+001 2.0190059e-002
2.9400000e+001 2.4676740e-002
2.9420000e+001 1.4377770e-002
2.9440000e+001 5.6083498e-003
2.9460000e+001 -6.7300197e-003
2.9480000e+001 -9.8910895e-003
2.9500000e+001 -2.6512198e-003
2.9520000e+001 1.6315200e-003
2.9540000e+001 1.0197000e-002
2.9560000e+001 9.5851793e-003
2.9580000e+001 3.9768300e-003
2.9600000e+001 0.0000000e+000
2.9620000e+001 -6.7300197e-003
2.9640000e+001 -1.0095029e-002
2.9660000e+001 -1.1114729e-002
2.9680000e+001 -1.1726550e-002
2.9700000e+001 -1.3358070e-002
2.9720000e+001 -1.7130959e-002
2.9740000e+001 -2.0292029e-002
2.9760000e+001 -1.0604879e-002
2.9780000e+001 -1.0197000e-003
2.9800000e+001 5.6083498e-003
2.9820000e+001 1.3154129e-002
2.9840000e+001 2.0393999e-002
2.9860000e+001 2.0292029e-002
2.9880000e+001 1.7232929e-002
2.9900000e+001 1.4581710e-002
2.9920000e+001 1.1114729e-002
2.9940000e+001 1.3052160e-002
2.9960000e+001 1.5499439e-002
2.9980000e+001 1.3969889e-002
3.0000000e+001 1.1930489e-002
3.0020000e+001 9.8910895e-003
3.0040000e+001 6.1181998e-003
3.0060000e+001 -2.8551599e-003
3.0080000e+001 -5.0984998e-003
3.0100000e+001 -8.5654793e-003
3.0120000e+001 -1.8150659e-002
3.0140000e+001 -3.2732367e-002
3.0160000e+001 -4.2827399e-002
3.0180000e+001 -5.2820457e-002
3.0200000e+001 -4.8129839e-002
3.0220000e+001 -4.0176180e-002
3.0240000e+001 -2.9571299e-002
3.0260000e+001 -1.1216700e-002
3.0280000e+001 5.0984998e-003
3.0300000e+001 1.3460039e-002
3.0320000e+001 1.6009289e-002
3.0340000e+001 1.8048690e-002
3.0360000e+001 1.9884149e-002
3.0380000e+001 2.6002349e-002
3.0400000e+001 3.3752067e-002
3.0420000e+001 3.3446158e-002
3.0440000e+001 2.5696440e-002
3.0460000e+001 1.7640809e-002
3.0480000e+001 4.5886498e-003
3.0500000e+001 -8.6674497e-003
3.0520000e+001 -1.9374299e-002
3.0540000e+001 -2.3351129e-002
3.0560000e+001 -3.0998879e-002
3.0580000e+001 -2.8245690e-002
3.0600000e+001 -2.3147190e-002
3.0620000e+001 -1.7742779e-002
3.0640000e+001 -1.2338370e-002
3.0660000e+001 -1.2848220e-002
3.0680000e+001 -1.3154129e-002
3.0700000e+001 -8.1575997e-003
3.0720000e+001 -2.6512198e-003
3.0740000e+001 3.9768300e-003
3.0760000e+001 6.6280498e-003
3.0780000e+001 5.6083498e-003
3.0800000e+001 5.0984998e-003
3.0820000e+001 6.5260799e-003
3.0840000e+001 1.2848220e-002
3.0860000e+001 1.8252629e-002
3.0880000e+001 2.4778708e-002
3.0900000e+001 3.1304790e-002
3.0920000e+001 3.0387058e-002
3.0940000e+001 2.5594470e-002
3.0960000e+001 2.2025520e-002
3.0980000e+001 1.6621109e-002
3.1000000e+001 1.9578240e-002
3.1020000e+001 2.3860979e-002
3.1040000e+001 2.8755540e-002
3.1060000e+001 3.0692969e-002
3.1080000e+001 1.9986120e-002
3.1100000e+001 1.0808820e-002
3.1120000e+001 -3.9768300e-003
3.1140000e+001 -1.6621109e-002
3.1160000e+001 -3.2834340e-002
3.1180000e+001 -3.4159949e-002
3.1200000e+001 -2.2331429e-002
3.1220000e+001 -1.5091560e-002
3.1240000e+001 -1.0197000e-003
3.1260000e+001 1.1216700e-003
3.1280000e+001 -5.4044100e-003
3.1300000e+001 -1.0298970e-002
3.1320000e+001 -1.2848220e-002
3.1340000e+001 -1.4581710e-002
3.1360000e+001 -1.3154129e-002
3.1380000e+001 -1.0604879e-002
3.1400000e+001 -7.2398697e-003
3.1420000e+001 -1.8354599e-003
3.1440000e+001 3.3650098e-003
3.1460000e+001 8.6674497e-003
3.1480000e+001 1.6111260e-002
3.1500000e+001 2.4370829e-002
3.1520000e+001 3.2528428e-002
3.1540000e+001 3.4771767e-002
3.1560000e+001 3.2426458e-002
3.1580000e+001 2.1719608e-002
3.1600000e+001 7.4438099e-003
3.1620000e+001 9.1772994e-004
3.1640000e+001 -1.5295499e-003
3.1660000e+001 -4.7925896e-003
3.1680000e+001 -7.6477497e-003
3.1700000e+001 -1.2338370e-002
3.1720000e+001 -1.5907320e-002
3.1740000e+001 -1.1828520e-002
3.1760000e+001 -5.7103197e-003
3.1780000e+001 -6.1182000e-004
3.1800000e+001 -3.0591000e-004
3.1820000e+001 -3.0591000e-004
3.1840000e+001 1.4275799e-003
3.1860000e+001 4.6906197e-003
3.1880000e+001 7.3418396e-003
3.1900000e+001 8.7694201e-003
3.1920000e+001 9.9930598e-003
3.1940000e+001 1.2644279e-002
3.1960000e+001 1.4989589e-002
3.1980000e+001 1.7538840e-002
3.2000000e+001 2.0393999e-002
3.2020000e+001 2.6104319e-002
3.2040000e+001 3.2324490e-002
3.2060000e+001 2.9265390e-002
3.2080000e+001 2.3555070e-002
3.2100000e+001 1.0706850e-002
3.2120000e+001 -1.0197000e-004
3.2140000e+001 -1.1216700e-003
3.2160000e+001 -3.6709198e-003
3.2180000e+001 -5.4044100e-003
3.2200000e+001 -8.4635099e-003
3.2220000e+001 -5.3024396e-003
3.2240000e+001 -7.1378996e-004
3.2260000e+001 3.7728899e-003
3.2280000e+001 9.7891200e-003
3.2300000e+001 1.5805349e-002
3.2320000e+001 2.0903849e-002
3.2340000e+001 1.4581710e-002
3.2360000e+001 7.4438099e-003
3.2380000e+001 2.1413698e-003
3.2400000e+001 -2.2433400e-003
3.2420000e+001 -7.1378998e-003
3.2440000e+001 -1.0197000e-002
3.2460000e+001 -7.6477497e-003
3.2480000e+001 -5.5063799e-003
3.2500000e+001 -2.9571300e-003
3.2520000e+001 -1.5295499e-003
3.2540000e+001 0.0000000e+000
3.2560000e+001 -1.0197000e-004
3.2580000e+001 -6.1182000e-004
3.2600000e+001 -1.1216700e-003
3.2620000e+001 -1.0197000e-003
3.2640000e+001 -3.0591000e-004
3.2660000e+001 1.0197000e-004
3.2680000e+001 1.6315200e-003
3.2700000e+001 5.4044100e-003
3.2720000e+001 8.7694201e-003
3.2740000e+001 1.2848220e-002
3.2760000e+001 1.5703379e-002
3.2780000e+001 1.3358070e-002
3.2800000e+001 1.0400939e-002
3.2820000e+001 5.7103197e-003
3.2840000e+001 6.1182000e-004
3.2860000e+001 -4.0787999e-003
3.2880000e+001 -9.9930598e-003
3.2900000e+001 -9.7891200e-003
3.2920000e+001 -4.6906197e-003
3.2940000e+001 -7.1378996e-004
3.2960000e+001 3.1610698e-003
3.2980000e+001 4.5886498e-003
3.3000000e+001 6.9339600e-003
3.3020000e+001 5.3024396e-003
3.3040000e+001 4.5886498e-003
3.3060000e+001 1.8354599e-003
3.3080000e+001 -2.0394000e-004
3.3100000e+001 -2.9571300e-003
3.3120000e+001 -2.8551599e-003
3.3140000e+001 -1.7334900e-003
3.3160000e+001 -6.1182000e-004
3.3180000e+001 -5.0984998e-004
3.3200000e+001 -1.7334900e-003
3.3220000e+001 -2.2433400e-003
3.3240000e+001 -3.6709198e-003
3.3260000e+001 -3.2630399e-003
3.3280000e+001 -7.1378996e-004
3.3300000e+001 1.4275799e-003
3.3320000e+001 4.1807698e-003
3.3340000e+001 6.6280498e-003
3.3360000e+001 6.4241100e-003
3.3380000e+001 5.3024396e-003
3.3400000e+001 6.1182000e-004
3.3420000e+001 -4.5886498e-003
3.3440000e+001 -7.7497196e-003
3.3460000e+001 -6.5260799e-003
3.3480000e+001 -6.6280498e-003
3.3500000e+001 -1.0910789e-002
3.3520000e+001 -1.6417170e-002
3.3540000e+001 -1.7742779e-002
3.3560000e+001 -1.1930489e-002
3.3580000e+001 -7.1378998e-003
3.3600000e+001 -3.6709198e-003
3.3620000e+001 -2.5492499e-003
3.3640000e+001 -2.0394000e-004
3.3660000e+001 4.0787999e-004
3.3680000e+001 3.6709198e-003
3.3700000e+001 8.3615395e-003
3.3720000e+001 1.1726550e-002
3.3740000e+001 7.1378998e-003
3.3760000e+001 3.8748598e-003
3.3780000e+001 -1.0197000e-003
3.3800000e+001 -2.5492499e-003
3.3820000e+001 -3.4669800e-003
3.3840000e+001 -4.1807698e-003
3.3860000e+001 -4.5886498e-003
3.3880000e+001 -6.6280498e-003
3.3900000e+001 -1.2236400e-002
3.3920000e+001 -1.1930489e-002
3.3940000e+001 -1.1012760e-002
3.3960000e+001 -9.9930598e-003
3.3980000e+001 -8.2595701e-003
3.4000000e+001 -7.7497196e-003
3.4020000e+001 -1.1216700e-002
3.4040000e+001 -1.4071860e-002
3.4060000e+001 -1.6927020e-002
3.4080000e+001 -1.4683679e-002
3.4100000e+001 -1.3256100e-002
3.4120000e+001 -1.1318670e-002
3.4140000e+001 -1.1012760e-002
3.4160000e+001 -9.9930598e-003
3.4180000e+001 -9.9930598e-003
3.4200000e+001 -6.4241100e-003
3.4220000e+001 -2.0394000e-004
3.4240000e+001 5.5063799e-003
3.4260000e+001 1.1216700e-002
3.4280000e+001 1.5499439e-002
3.4300000e+001 2.0088090e-002
3.4320000e+001 1.7334899e-002
3.4340000e+001 1.4275800e-002
3.4360000e+001 1.0502910e-002
3.4380000e+001 6.5260799e-003
3.4400000e+001 1.0197000e-003
3.4420000e+001 -6.8319896e-003
3.4440000e+001 -1.0706850e-002
3.4460000e+001 -1.2440339e-002
3.4480000e+001 -1.4071860e-002
3.4500000e+001 -1.6927020e-002
3.4520000e+001 -2.1923549e-002
3.4540000e+001 -2.7429929e-002
3.4560000e+001 -2.7939779e-002
3.4580000e+001 -2.2127490e-002
3.4600000e+001 -1.8150659e-002
3.4620000e+001 -1.1828520e-002
3.4640000e+001 -6.2201697e-003
3.4660000e+001 0.0000000e+000
3.4680000e+001 5.5063799e-003
3.4700000e+001 1.0095029e-002
3.4720000e+001 1.4683679e-002
3.4740000e+001 1.9068390e-002
3.4760000e+001 2.3657040e-002
3.4780000e+001 2.6716140e-002
3.4800000e+001 2.6002349e-002
3.4820000e+001 2.3147190e-002
3.4840000e+001 1.7436870e-002
3.4860000e+001 1.2950189e-002
3.4880000e+001 5.5063799e-003
3.4900000e+001 -4.7925896e-003
3.4920000e+001 -1.5193529e-002
3.4940000e+001 -2.1413699e-002
3.4960000e+001 -2.5900379e-002
3.4980000e+001 -2.5696440e-002
3.5000000e+001 -2.4982649e-002
3.5020000e+001 -2.3045220e-002
3.5040000e+001 -1.8252629e-002
3.5060000e+001 -1.3867920e-002
3.5080000e+001 -8.3615395e-003
3.5100000e+001 -1.4275799e-003
3.5120000e+001 4.4866799e-003
3.5140000e+001 9.0753293e-003
3.5160000e+001 9.8910895e-003
3.5180000e+001 1.0197000e-002
3.5200000e+001 1.0197000e-002
3.5220000e+001 1.0604879e-002
3.5240000e+001 1.3867920e-002
3.5260000e+001 1.2338370e-002
3.5280000e+001 1.0197000e-002
3.5300000e+001 7.3418396e-003
3.5320000e+001 1.9374299e-003
3.5340000e+001 -2.8551599e-003
3.5360000e+001 -8.4635099e-003
3.5380000e+001 -1.3358070e-002
3.5400000e+001 -1.9374299e-002
3.5420000e+001 -1.9782179e-002
3.5440000e+001 -1.4173829e-002
3.5460000e+001 -1.0197000e-002
3.5480000e+001 -4.4866799e-003
3.5500000e+001 -4.4866799e-003
3.5520000e+001 -4.0787999e-003
3.5540000e+001 -4.5886498e-003
3.5560000e+001 -4.9965299e-003
3.5580000e+001 -3.9768300e-003
3.5600000e+001 -2.1413698e-003
3.5620000e+001 -2.2433400e-003
3.5640000e+001 -1.6315200e-003
3.5660000e+001 -1.6315200e-003
3.5680000e+001 -1.2236400e-003
3.5700000e+001 -1.2236400e-003
3.5720000e+001 2.8551599e-003
3.5740000e+001 6.9339600e-003
3.5760000e+001 1.1726550e-002
3.5780000e+001 9.7891200e-003
3.5800000e+001 5.7103197e-003
3.5820000e+001 1.7334900e-003
3.5840000e+001 -3.1610698e-003
3.5860000e+001 -7.9536599e-003
3.5880000e+001 -9.3812395e-003
3.5900000e+001 -9.3812395e-003
3.5920000e+001 -9.6871497e-003
3.5940000e+001 -9.3812395e-003
3.5960000e+001 -8.7694201e-003
3.5980000e+001 -5.3024396e-003
3.6000000e+001 -2.2433400e-003
3.6020000e+001 1.4275799e-003
3.6040000e+001 3.8748598e-003
3.6060000e+001 3.4669800e-003
3.6080000e+001 3.4669800e-003
3.6100000e+001 2.9571300e-003
3.6120000e+001 4.0787999e-003
3.6140000e+001 5.5063799e-003
3.6160000e+001 6.8319896e-003
3.6180000e+001 8.6674497e-003
3.6200000e+001 8.8713895e-003
3.6220000e+001 7.8516895e-003
3.6240000e+001 7.1378998e-003
3.6260000e+001 6.2201697e-003
3.6280000e+001 7.6477497e-003
3.6300000e+001 8.8713895e-003
3.6320000e+001 1.0400939e-002
3.6340000e+001 1.1318670e-002
3.6360000e+001 9.1772997e-003
3.6380000e+001 7.6477497e-003
3.6400000e+001 5.4044100e-003
3.6420000e+001 6.3221396e-003
3.6440000e+001 8.0556298e-003
3.6460000e+001 9.8910895e-003
3.6480000e+001 1.2032460e-002
3.6500000e+001 1.3765950e-002
3.6520000e+001 1.0706850e-002
3.6540000e+001 7.8516895e-003
3.6560000e+001 4.4866799e-003
3.6580000e+001 1.2236400e-003
3.6600000e+001 -2.1413698e-003
3.6620000e+001 -4.2827397e-003
3.6640000e+001 -6.9339600e-003
3.6660000e+001 -8.8713895e-003
3.6680000e+001 -1.2236400e-002
3.6700000e+001 -1.6213229e-002
3.6720000e+001 -2.0190059e-002
3.6740000e+001 -2.3759008e-002
3.6760000e+001 -2.6308258e-002
3.6780000e+001 -2.7531899e-002
3.6800000e+001 -2.8347658e-002
3.6820000e+001 -2.9265390e-002
3.6840000e+001 -2.8347658e-002
3.6860000e+001 -2.7531899e-002
3.6880000e+001 -2.6308258e-002
3.6900000e+001 -2.5594470e-002
3.6920000e+001 -2.2331429e-002
3.6940000e+001 -1.7640809e-002
3.6960000e+001 -1.3154129e-002
3.6980000e+001 -1.1114729e-002
3.7000000e+001 -1.0706850e-002
3.7020000e+001 -9.6871497e-003
3.7040000e+001 -9.2792701e-003
3.7060000e+001 -7.3418396e-003
3.7080000e+001 -6.0162299e-003
3.7100000e+001 -4.0787999e-003
3.7120000e+001 -2.3453099e-003
3.7140000e+001 1.8354599e-003
3.7160000e+001 3.5689499e-003
3.7180000e+001 4.2827397e-003
3.7200000e+001 4.8945600e-003
3.7220000e+001 4.3847100e-003
3.7240000e+001 4.3847100e-003
3.7260000e+001 3.8748598e-003
3.7280000e+001 3.6709198e-003
3.7300000e+001 3.2630399e-003
3.7320000e+001 4.7925896e-003
3.7340000e+001 6.4241100e-003
3.7360000e+001 8.6674497e-003
3.7380000e+001 1.1216700e-002
3.7400000e+001 1.3663979e-002
3.7420000e+001 1.6315199e-002
3.7440000e+001 1.8456570e-002
3.7460000e+001 1.7640809e-002
3.7480000e+001 1.7130959e-002
3.7500000e+001 1.6213229e-002
3.7520000e+001 1.4989589e-002
3.7540000e+001 1.1012760e-002
3.7560000e+001 6.9339600e-003
3.7580000e+001 3.2630399e-003
3.7600000e+001 9.1772994e-004
3.7620000e+001 1.9374299e-003
3.7640000e+001 2.8551599e-003
3.7660000e+001 4.0787999e-003
3.7680000e+001 5.5063799e-003
3.7700000e+001 7.0359299e-003
3.7720000e+001 9.6871497e-003
3.7740000e+001 1.1930489e-002
3.7760000e+001 1.4581710e-002
3.7780000e+001 1.4581710e-002
3.7800000e+001 1.2746250e-002
3.7820000e+001 1.2440339e-002
3.7840000e+001 1.2644279e-002
3.7860000e+001 1.2848220e-002
3.7880000e+001 1.3256100e-002
3.7900000e+001 1.3562010e-002
3.7920000e+001 1.0808820e-002
3.7940000e+001 8.0556298e-003
3.7960000e+001 4.6906197e-003
3.7980000e+001 2.0393999e-003
3.8000000e+001 5.0984998e-004
3.8020000e+001 -1.1216700e-003
3.8040000e+001 -2.5492499e-003
3.8060000e+001 -1.1216700e-003
3.8080000e+001 1.8354599e-003
3.8100000e+001 4.3847100e-003
3.8120000e+001 7.7497196e-003
3.8140000e+001 1.0502910e-002
3.8160000e+001 1.3562010e-002
3.8180000e+001 1.0298970e-002
3.8200000e+001 6.3221396e-003
3.8220000e+001 1.8354599e-003
3.8240000e+001 -2.7531900e-003
3.8260000e+001 -8.1575997e-003
3.8280000e+001 -9.0753293e-003
3.8300000e+001 -5.5063799e-003
3.8320000e+001 -3.0590999e-003
3.8340000e+001 6.1182000e-004
3.8360000e+001 1.2236400e-003
3.8380000e+001 2.2433400e-003
3.8400000e+001 2.7531900e-003
3.8420000e+001 3.2630399e-003
3.8440000e+001 4.8945600e-003
3.8460000e+001 7.0359299e-003
3.8480000e+001 9.0753293e-003
3.8500000e+001 1.1420639e-002
3.8520000e+001 1.3562010e-002
3.8540000e+001 1.4275800e-002
3.8560000e+001 1.3969889e-002
3.8580000e+001 1.4275800e-002
3.8600000e+001 1.3765950e-002
3.8620000e+001 1.4173829e-002
3.8640000e+001 1.0400939e-002
3.8660000e+001 3.0590999e-003
3.8680000e+001 -3.2630399e-003
3.8700000e+001 -6.4241100e-003
3.8720000e+001 -8.6674497e-003
3.8740000e+001 -1.1420639e-002
3.8760000e+001 -1.6009289e-002
3.8780000e+001 -2.0088090e-002
3.8800000e+001 -1.9170359e-002
3.8820000e+001 -1.8558540e-002
3.8840000e+001 -1.7334899e-002
3.8860000e+001 -1.5907320e-002
3.8880000e+001 -1.4479739e-002
3.8900000e+001 -1.3154129e-002
3.8920000e+001 -1.2746250e-002
3.8940000e+001 -1.2236400e-002
3.8960000e+001 -1.1828520e-002
3.8980000e+001 -1.1624579e-002
3.9000000e+001 -1.0808820e-002
3.9020000e+001 -6.5260799e-003
3.9040000e+001 -2.1413698e-003
3.9060000e+001 2.0394000e-004
3.9080000e+001 -4.0787999e-004
3.9100000e+001 -4.0787999e-004
3.9120000e+001 -1.4275799e-003
3.9140000e+001 -1.9374299e-003
3.9160000e+001 -8.1575998e-004
3.9180000e+001 1.1216700e-003
3.9200000e+001 2.7531900e-003
3.9220000e+001 5.0984998e-003
3.9240000e+001 5.4044100e-003
3.9260000e+001 4.2827397e-003
3.9280000e+001 3.0590999e-003
3.9300000e+001 1.1216700e-003
3.9320000e+001 -5.0984998e-004
3.9340000e+001 -2.4472800e-003
3.9360000e+001 -2.0393999e-003
3.9380000e+001 -7.1378996e-004
3.9400000e+001 4.0787999e-004
3.9420000e+001 1.9374299e-003
3.9440000e+001 3.3650098e-003
3.9460000e+001 4.8945600e-003
3.9480000e+001 5.8122896e-003
3.9500000e+001 6.6280498e-003
3.9520000e+001 7.5457798e-003
3.9540000e+001 8.2595701e-003
3.9560000e+001 9.2792701e-003
3.9580000e+001 1.2950189e-002
3.9600000e+001 1.7232929e-002
3.9620000e+001 1.4683679e-002
3.9640000e+001 1.0706850e-002
3.9660000e+001 7.1378998e-003
3.9680000e+001 3.5689499e-003
3.9700000e+001 2.0394000e-004
3.9720000e+001 -3.2630399e-003
3.9740000e+001 -6.5260799e-003
3.9760000e+001 -9.6871497e-003
3.9780000e+001 -9.2792701e-003
3.9800000e+001 -8.4635099e-003
3.9820000e+001 -7.4438099e-003
3.9840000e+001 -5.2004697e-003
3.9860000e+001 -3.0590999e-003
3.9880000e+001 -4.0787999e-004
3.9900000e+001 3.3650098e-003
3.9920000e+001 5.3024396e-003
3.9940000e+001 4.4866799e-003
3.9960000e+001 4.0787999e-003
3.9980000e+001 3.0590999e-003
4.0000000e+001 2.0393999e-003
4.0020000e+001 1.0197000e-003
4.0040000e+001 1.0197000e-004
4.0060000e+001 -1.0197000e-003
4.0080000e+001 -1.4275799e-003
4.0100000e+001 2.0394000e-004
4.0120000e+001 1.6315200e-003
4.0140000e+001 3.0590999e-003
4.0160000e+001 2.3453099e-003
4.0180000e+001 1.6315200e-003
4.0200000e+001 1.6315200e-003
4.0220000e+001 2.6512198e-003
4.0240000e+001 3.2630399e-003
4.0260000e+001 4.5886498e-003
4.0280000e+001 4.6906197e-003
4.0300000e+001 2.1413698e-003
4.0320000e+001 -3.0591000e-004
4.0340000e+001 -3.0590999e-003
4.0360000e+001 -6.1181998e-003
4.0380000e+001 -8.8713895e-003
4.0400000e+001 -1.1828520e-002
4.0420000e+001 -1.0910789e-002
4.0440000e+001 -9.5851793e-003
4.0460000e+001 -7.9536599e-003
4.0480000e+001 -6.9339600e-003
4.0500000e+001 -6.4241100e-003
4.0520000e+001 -5.8122896e-003
4.0540000e+001 -5.0984998e-003
4.0560000e+001 -4.1807698e-003
4.0580000e+001 -3.3650098e-003
4.0600000e+001 -2.4472800e-003
4.0620000e+001 -1.4275799e-003
4.0640000e+001 -5.0984998e-004
4.0660000e+001 4.0787999e-004
4.0680000e+001 1.3256099e-003
4.0700000e+001 2.1413698e-003
4.0720000e+001 1.0197000e-004
4.0740000e+001 -1.8354599e-003
4.0760000e+001 -4.0787999e-003
4.0780000e+001 -6.5260799e-003
4.0800000e+001 -8.8713895e-003
4.0820000e+001 -1.1216700e-002
4.0840000e+001 -1.3154129e-002
4.0860000e+001 -1.5397470e-002
4.0880000e+001 -1.6825049e-002
4.0900000e+001 -1.7742779e-002
4.0920000e+001 -1.8762479e-002
4.0940000e+001 -1.9782179e-002
4.0960000e+001 -2.0597940e-002
4.0980000e+001 -2.1719608e-002
4.1000000e+001 -2.0190059e-002
4.1020000e+001 -1.7640809e-002
4.1040000e+001 -1.3765950e-002
4.1060000e+001 -6.6280498e-003
4.1080000e+001 -1.4275799e-003
4.1100000e+001 2.0393999e-003
4.1120000e+001 5.6083498e-003
4.1140000e+001 8.3615395e-003
4.1160000e+001 9.0753293e-003
4.1180000e+001 1.0400939e-002
4.1200000e+001 9.4832099e-003
4.1220000e+001 8.1575997e-003
4.1240000e+001 6.2201697e-003
4.1260000e+001 3.3650098e-003
4.1280000e+001 3.0591000e-004
4.1300000e+001 -2.4472800e-003
4.1320000e+001 -5.8122896e-003
4.1340000e+001 -4.5886498e-003
4.1360000e+001 -1.3256099e-003
4.1380000e+001 1.3256099e-003
4.1400000e+001 3.3650098e-003
4.1420000e+001 4.8945600e-003
4.1440000e+001 5.5063799e-003
4.1460000e+001 5.9142600e-003
4.1480000e+001 6.3221396e-003
4.1500000e+001 6.6280498e-003
4.1520000e+001 7.0359299e-003
4.1540000e+001 7.3418396e-003
4.1560000e+001 7.6477497e-003
4.1580000e+001 7.8516895e-003
4.1600000e+001 7.4438099e-003
4.1620000e+001 7.0359299e-003
4.1640000e+001 6.5260799e-003
4.1660000e+001 5.9142600e-003
4.1680000e+001 5.5063799e-003
4.1700000e+001 4.8945600e-003
4.1720000e+001 6.2201697e-003
4.1740000e+001 8.0556298e-003
4.1760000e+001 9.7891200e-003
4.1780000e+001 1.1624579e-002
4.1800000e+001 1.2848220e-002
4.1820000e+001 1.4173829e-002
4.1840000e+001 1.3358070e-002
4.1860000e+001 1.2542310e-002
4.1880000e+001 1.1624579e-002
4.1900000e+001 1.0502910e-002
4.1920000e+001 8.8713895e-003
4.1940000e+001 7.3418396e-003
4.1960000e+001 5.7103197e-003
4.1980000e+001 4.0787999e-003
4.2000000e+001 2.4472800e-003
4.2020000e+001 7.1378996e-004
4.2040000e+001 4.0787999e-004
4.2060000e+001 8.1575998e-004
4.2080000e+001 9.1772994e-004
4.2100000e+001 2.3453099e-003
4.2120000e+001 4.2827397e-003
4.2140000e+001 5.3024396e-003
4.2160000e+001 6.2201697e-003
4.2180000e+001 7.3418396e-003
4.2200000e+001 8.0556298e-003
4.2220000e+001 9.0753293e-003
4.2240000e+001 9.6871497e-003
4.2260000e+001 9.6871497e-003
4.2280000e+001 9.4832099e-003
4.2300000e+001 9.0753293e-003
4.2320000e+001 2.5492499e-003
4.2340000e+001 0.0000000e+000
4.2360000e+001 -5.0984998e-004
4.2380000e+001 -2.0393999e-003
4.2400000e+001 -2.4472800e-003
4.2420000e+001 -3.3650098e-003
4.2440000e+001 -4.8945600e-003
4.2460000e+001 -6.5260799e-003
4.2480000e+001 -8.1575997e-003
4.2500000e+001 -9.8910895e-003
4.2520000e+001 -1.1624579e-002
4.2540000e+001 -1.3052160e-002
4.2560000e+001 -1.0808820e-002
4.2580000e+001 -9.0753293e-003
4.2600000e+001 -6.7300197e-003
4.2620000e+001 -4.3847100e-003
4.2640000e+001 -1.9374299e-003
4.2660000e+001 -1.1216700e-003
4.2680000e+001 -9.1772994e-004
4.2700000e+001 -5.0984998e-004
4.2720000e+001 -4.0787999e-004
4.2740000e+001 5.0984998e-004
4.2760000e+001 1.4275799e-003
4.2780000e+001 2.4472800e-003
4.2800000e+001 3.5689499e-003
4.2820000e+001 4.3847100e-003
4.2840000e+001 4.4866799e-003
4.2860000e+001 3.9768300e-003
4.2880000e+001 1.3256099e-003
4.2900000e+001 -8.1575998e-004
4.2920000e+001 -3.6709198e-003
4.2940000e+001 -4.7925896e-003
4.2960000e+001 -5.6083498e-003
4.2980000e+001 -6.3221396e-003
4.3000000e+001 -6.9339600e-003
4.3020000e+001 -7.4438099e-003
4.3040000e+001 -8.1575997e-003
4.3060000e+001 -7.6477497e-003
4.3080000e+001 -6.5260799e-003
4.3100000e+001 -2.8551599e-003
4.3120000e+001 5.0984998e-004
4.3140000e+001 4.6906197e-003
4.3160000e+001 6.6280498e-003
4.3180000e+001 6.1181998e-003
4.3200000e+001 6.1181998e-003
4.3220000e+001 5.5063799e-003
4.3240000e+001 4.9965299e-003
4.3260000e+001 4.4866799e-003
4.3280000e+001 3.9768300e-003
4.3300000e+001 2.9571300e-003
4.3320000e+001 0.0000000e+000
4.3340000e+001 -2.6512198e-003
4.3360000e+001 -5.3024396e-003
4.3380000e+001 -4.2827397e-003
4.3400000e+001 -3.7728899e-003
4.3420000e+001 -2.5492499e-003
4.3440000e+001 -1.5295499e-003
4.3460000e+001 -2.0394000e-004
4.3480000e+001 2.0394000e-004
4.3500000e+001 0.0000000e+000
4.3520000e+001 -1.0197000e-004
4.3540000e+001 -4.0787999e-004
4.3560000e+001 -7.1378996e-004
4.3580000e+001 -6.1182000e-004
4.3600000e+001 -1.0197000e-004
4.3620000e+001 3.0591000e-004
4.3640000e+001 7.1378996e-004
4.3660000e+001 -1.0197000e-004
4.3680000e+001 -7.1378996e-004
4.3700000e+001 -1.5295499e-003
4.3720000e+001 -2.4472800e-003
4.3740000e+001 -2.9571300e-003
4.3760000e+001 -2.8551599e-003
4.3780000e+001 -2.8551599e-003
4.3800000e+001 -2.5492499e-003
4.3820000e+001 -2.4472800e-003
4.3840000e+001 -2.1413698e-003
4.3860000e+001 -2.7531900e-003
4.3880000e+001 -4.4866799e-003
4.3900000e+001 -5.9142600e-003
4.3920000e+001 -7.7497196e-003
4.3940000e+001 -8.5654793e-003
4.3960000e+001 -8.3615395e-003
4.3980000e+001 -8.4635099e-003
4.4000000e+001 -7.6477497e-003
4.4020000e+001 -5.0984998e-003
4.4040000e+001 -2.8551599e-003
4.4060000e+001 -2.0394000e-004
4.4080000e+001 1.8354599e-003
4.4100000e+001 1.8354599e-003
4.4120000e+001 2.1413698e-003
4.4140000e+001 2.1413698e-003
4.4160000e+001 2.1413698e-003
4.4180000e+001 3.1610698e-003
4.4200000e+001 4.3847100e-003
4.4220000e+001 4.9965299e-003
4.4240000e+001 4.9965299e-003
4.4260000e+001 4.9965299e-003
4.4280000e+001 4.8945600e-003
4.4300000e+001 4.7925896e-003
4.4320000e+001 5.3024396e-003
4.4340000e+001 6.8319896e-003
4.4360000e+001 8.0556298e-003
4.4380000e+001 9.6871497e-003
4.4400000e+001 9.8910895e-003
4.4420000e+001 9.7891200e-003
4.4440000e+001 9.7891200e-003
4.4460000e+001 9.4832099e-003
4.4480000e+001 9.2792701e-003
4.4500000e+001 9.0753293e-003
4.4520000e+001 8.8713895e-003
4.4540000e+001 8.5654793e-003
4.4560000e+001 8.3615395e-003
4.4580000e+001 8.1575997e-003
4.4600000e+001 7.8516895e-003
4.4620000e+001 6.3221396e-003
4.4640000e+001 4.7925896e-003
4.4660000e+001 3.5689499e-003
4.4680000e+001 3.6709198e-003
4.4700000e+001 3.4669800e-003
4.4720000e+001 3.5689499e-003
4.4740000e+001 3.6709198e-003
4.4760000e+001 3.2630399e-003
4.4780000e+001 1.9374299e-003
4.4800000e+001 1.5295499e-003
4.4820000e+001 1.4275799e-003
4.4840000e+001 1.2236400e-003
4.4860000e+001 1.2236400e-003
4.4880000e+001 1.1216700e-003
4.4900000e+001 0.0000000e+000
4.4920000e+001 -1.0197000e-003
4.4940000e+001 -2.0393999e-003
4.4960000e+001 -2.5492499e-003
4.4980000e+001 -2.9571300e-003
4.5000000e+001 -3.3650098e-003
4.5020000e+001 -3.7728899e-003
4.5040000e+001 -4.8945600e-003
4.5060000e+001 -6.4241100e-003
4.5080000e+001 -7.8516895e-003
4.5100000e+001 -9.3812395e-003
4.5120000e+001 -1.0910789e-002
4.5140000e+001 -1.2440339e-002
4.5160000e+001 -1.4071860e-002
4.5180000e+001 -1.4581710e-002
4.5200000e+001 -1.2236400e-002
4.5220000e+001 -1.0502910e-002
4.5240000e+001 -7.9536599e-003
4.5260000e+001 -6.8319896e-003
4.5280000e+001 -6.4241100e-003
4.5300000e+001 -5.9142600e-003
4.5320000e+001 -5.7103197e-003
4.5340000e+001 -5.4044100e-003
4.5360000e+001 -5.0984998e-003
4.5380000e+001 -4.8945600e-003
4.5400000e+001 -4.5886498e-003
4.5420000e+001 -4.2827397e-003
4.5440000e+001 -3.9768300e-003
4.5460000e+001 -3.1610698e-003
4.5480000e+001 -2.6512198e-003
4.5500000e+001 -1.1216700e-003
4.5520000e+001 1.0197000e-003
4.5540000e+001 3.1610698e-003
4.5560000e+001 4.6906197e-003
4.5580000e+001 4.8945600e-003
4.5600000e+001 5.5063799e-003
4.5620000e+001 5.6083498e-003
4.5640000e+001 6.4241100e-003
4.5660000e+001 8.5654793e-003
4.5680000e+001 1.0298970e-002
4.5700000e+001 8.5654793e-003
4.5720000e+001 7.2398697e-003
4.5740000e+001 5.4044100e-003
4.5760000e+001 3.3650098e-003
4.5780000e+001 2.8551599e-003
4.5800000e+001 2.9571300e-003
4.5820000e+001 2.8551599e-003
4.5840000e+001 2.7531900e-003
4.5860000e+001 1.2236400e-003
4.5880000e+001 -1.0197000e-004
4.5900000e+001 -1.0197000e-003
4.5920000e+001 -1.1216700e-003
4.5940000e+001 -1.6315200e-003
4.5960000e+001 -1.5295499e-003
4.5980000e+001 -3.1610698e-003
4.6000000e+001 -6.9339600e-003
4.6020000e+001 -1.0095029e-002
4.6040000e+001 -1.0400939e-002
4.6060000e+001 -1.0706850e-002
4.6080000e+001 -1.0808820e-002
4.6100000e+001 -1.0095029e-002
4.6120000e+001 -9.0753293e-003
4.6140000e+001 -8.0556298e-003
4.6160000e+001 -7.5457798e-003
4.6180000e+001 -7.3418396e-003
4.6200000e+001 -6.9339600e-003
4.6220000e+001 -6.8319896e-003
4.6240000e+001 -6.4241100e-003
4.6260000e+001 -4.0787999e-003
4.6280000e+001 -1.3256099e-003
4.6300000e+001 1.3256099e-003
4.6320000e+001 4.3847100e-003
4.6340000e+001 7.2398697e-003
4.6360000e+001 1.0095029e-002
4.6380000e+001 8.7694201e-003
4.6400000e+001 6.4241100e-003
4.6420000e+001 4.1807698e-003
4.6440000e+001 1.3256099e-003
4.6460000e+001 -1.4275799e-003
4.6480000e+001 -3.5689499e-003
4.6500000e+001 -3.7728899e-003
4.6520000e+001 -4.4866799e-003
4.6540000e+001 -4.6906197e-003
4.6560000e+001 -4.9965299e-003
4.6580000e+001 -5.2004697e-003
4.6600000e+001 -5.5063799e-003
4.6620000e+001 -5.7103197e-003
4.6640000e+001 -5.9142600e-003
4.6660000e+001 -4.6906197e-003
4.6680000e+001 -2.9571300e-003
4.6700000e+001 -1.3256099e-003
4.6720000e+001 7.1378996e-004
4.6740000e+001 1.7334900e-003
4.6760000e+001 1.9374299e-003
4.6780000e+001 2.5492499e-003
4.6800000e+001 2.5492499e-003
4.6820000e+001 3.5689499e-003
4.6840000e+001 5.0984998e-003
4.6860000e+001 6.5260799e-003
4.6880000e+001 8.1575997e-003
4.6900000e+001 9.5851793e-003
4.6920000e+001 9.8910895e-003
4.6940000e+001 1.0400939e-002
4.6960000e+001 1.0706850e-002
4.6980000e+001 1.1012760e-002
4.7000000e+001 1.1216700e-002
4.7020000e+001 1.0604879e-002
4.7040000e+001 1.0095029e-002
4.7060000e+001 9.4832099e-003
4.7080000e+001 8.7694201e-003
4.7100000e+001 8.0556298e-003
4.7120000e+001 7.3418396e-003
4.7140000e+001 6.4241100e-003
4.7160000e+001 5.6083498e-003
4.7180000e+001 4.6906197e-003
4.7200000e+001 3.8748598e-003
4.7220000e+001 3.1610698e-003
4.7240000e+001 2.7531900e-003
4.7260000e+001 2.1413698e-003
4.7280000e+001 1.7334900e-003
4.7300000e+001 1.2236400e-003
4.7320000e+001 7.1378996e-004
4.7340000e+001 3.0591000e-004
4.7360000e+001 8.1575998e-004
4.7380000e+001 1.3256099e-003
4.7400000e+001 1.9374299e-003
4.7420000e+001 2.2433400e-003
4.7440000e+001 9.1772994e-004
4.7460000e+001 -3.0591000e-004
4.7480000e+001 -1.7334900e-003
4.7500000e+001 -3.1610698e-003
4.7520000e+001 -4.7925896e-003
4.7540000e+001 -5.4044100e-003
4.7560000e+001 -4.8945600e-003
4.7580000e+001 -4.3847100e-003
4.7600000e+001 -3.8748598e-003
4.7620000e+001 -2.5492499e-003
4.7640000e+001 -9.1772994e-004
4.7660000e+001 7.1378996e-004
4.7680000e+001 1.7334900e-003
4.7700000e+001 2.2433400e-003
4.7720000e+001 3.5689499e-003
4.7740000e+001 6.2201697e-003
4.7760000e+001 8.4635099e-003
4.7780000e+001 1.1420639e-002
4.7800000e+001 1.0910789e-002
4.7820000e+001 8.8713895e-003
4.7840000e+001 7.2398697e-003
4.7860000e+001 4.7925896e-003
4.7880000e+001 2.7531900e-003
4.7900000e+001 4.0787999e-004
4.7920000e+001 -1.6315200e-003
4.7940000e+001 -2.4472800e-003
4.7960000e+001 -3.3650098e-003
4.7980000e+001 -4.6906197e-003
4.8000000e+001 -7.0359299e-003
4.8020000e+001 -8.5654793e-003
4.8040000e+001 -8.9733599e-003
4.8060000e+001 -9.4832099e-003
4.8080000e+001 -9.6871497e-003
4.8100000e+001 -9.8910895e-003
4.8120000e+001 -1.0095029e-002
4.8140000e+001 -1.0298970e-002
4.8160000e+001 -1.0400939e-002
4.8180000e+001 -9.9930598e-003
4.8200000e+001 -9.6871497e-003
4.8220000e+001 -9.1772997e-003
4.8240000e+001 -8.8713895e-003
4.8260000e+001 -9.0753293e-003
4.8280000e+001 -9.2792701e-003
4.8300000e+001 -9.4832099e-003
4.8320000e+001 -9.7891200e-003
4.8340000e+001 -9.8910895e-003
4.8360000e+001 -8.9733599e-003
4.8380000e+001 -8.2595701e-003
4.8400000e+001 -7.2398697e-003
4.8420000e+001 -6.9339600e-003
4.8440000e+001 -6.9339600e-003
4.8460000e+001 -6.8319896e-003
4.8480000e+001 -6.8319896e-003
4.8500000e+001 -6.7300197e-003
4.8520000e+001 -6.7300197e-003
4.8540000e+001 -6.5260799e-003
4.8560000e+001 -5.7103197e-003
4.8580000e+001 -5.0984998e-003
4.8600000e+001 -4.3847100e-003
4.8620000e+001 -3.4669800e-003
4.8640000e+001 -2.3453099e-003
4.8660000e+001 -1.3256099e-003
4.8680000e+001 -2.0394000e-004
4.8700000e+001 1.0197000e-003
4.8720000e+001 1.7334900e-003
4.8740000e+001 9.1772994e-004
4.8760000e+001 3.0591000e-004
4.8780000e+001 -5.0984998e-004
4.8800000e+001 -1.4275799e-003
4.8820000e+001 -7.1378996e-004
4.8840000e+001 6.1182000e-004
4.8860000e+001 1.6315200e-003
4.8880000e+001 1.3256099e-003
4.8900000e+001 1.2236400e-003
4.8920000e+001 8.1575998e-004
4.8940000e+001 5.0984998e-004
4.8960000e+001 0.0000000e+000
4.8980000e+001 2.0394000e-004
4.9000000e+001 9.1772994e-004
4.9020000e+001 1.4275799e-003
4.9040000e+001 2.1413698e-003
4.9060000e+001 2.2433400e-003
4.9080000e+001 1.0197000e-003
4.9100000e+001 0.0000000e+000
4.9120000e+001 -1.2236400e-003
4.9140000e+001 -2.4472800e-003
4.9160000e+001 -2.4472800e-003
4.9180000e+001 -1.9374299e-003
4.9200000e+001 -1.6315200e-003
4.9220000e+001 -1.8354599e-003
4.9240000e+001 -2.2433400e-003
4.9260000e+001 -2.7531900e-003
4.9280000e+001 -3.0590999e-003
4.9300000e+001 -2.6512198e-003
4.9320000e+001 -2.2433400e-003
4.9340000e+001 -2.2433400e-003
4.9360000e+001 -3.2630399e-003
4.9380000e+001 -4.1807698e-003
4.9400000e+001 -5.2004697e-003
4.9420000e+001 -4.6906197e-003
4.9440000e+001 -4.3847100e-003
4.9460000e+001 -3.8748598e-003
4.9480000e+001 -3.2630399e-003
4.9500000e+001 -2.6512198e-003
4.9520000e+001 -2.0393999e-003
4.9540000e+001 -1.9374299e-003
4.9560000e+001 -1.8354599e-003
4.9580000e+001 -1.5295499e-003
4.9600000e+001 1.3256099e-003
4.9620000e+001 4.1807698e-003
4.9640000e+001 6.8319896e-003
4.9660000e+001 7.1378998e-003
4.9680000e+001 7.7497196e-003
4.9700000e+001 7.9536599e-003
4.9720000e+001 8.1575997e-003
4.9740000e+001 8.2595701e-003
4.9760000e+001 8.5654793e-003
4.9780000e+001 7.9536599e-003
4.9800000e+001 6.2201697e-003
4.9820000e+001 4.8945600e-003
4.9840000e+001 3.1610698e-003
4.9860000e+001 2.9571300e-003
4.9880000e+001 3.0590999e-003
4.9900000e+001 3.2630399e-003
4.9920000e+001 3.5689499e-003
4.9940000e+001 3.7728899e-003
4.9960000e+001 2.4472800e-003
4.9980000e+001 7.1378996e-004
5.0000000e+001 -8.1575998e-004
5.0020000e+001 -2.8551599e-003
5.0040000e+001 -3.5689499e-003
5.0060000e+001 -2.2433400e-003
5.0080000e+001 -1.3256099e-003
5.0100000e+001 1.0197000e-004
5.0120000e+001 3.0591000e-004
5.0140000e+001 1.0197000e-004
5.0160000e+001 -1.0197000e-004
5.0180000e+001 -5.0984998e-004
5.0200000e+001 -8.1575998e-004
5.0220000e+001 -1.1216700e-003
5.0240000e+001 -1.5295499e-003
5.0260000e+001 -1.2236400e-003
5.0280000e+001 -8.1575998e-004
5.0300000e+001 -4.0787999e-004
5.0320000e+001 1.0197000e-004
5.0340000e+001 5.0984998e-004
5.0360000e+001 8.1575998e-004
5.0380000e+001 8.1575998e-004
5.0400000e+001 9.1772994e-004
5.0420000e+001 1.0197000e-003
5.0440000e+001 2.2433400e-003
5.0460000e+001 3.3650098e-003
5.0480000e+001 4.6906197e-003
5.0500000e+001 6.0162299e-003
5.0520000e+001 7.3418396e-003
5.0540000e+001 8.0556298e-003
5.0560000e+001 6.3221396e-003
5.0580000e+001 5.0984998e-003
5.0600000e+001 3.2630399e-003
5.0620000e+001 1.6315200e-003
5.0640000e+001 -4.0787999e-004
5.0660000e+001 0.0000000e+000
5.0680000e+001 3.0590999e-003
5.0700000e+001 5.3024396e-003
5.0720000e+001 8.8713895e-003
5.0740000e+001 8.3615395e-003
5.0760000e+001 3.1610698e-003
5.0780000e+001 -2.0394000e-004
5.0800000e+001 -6.3221396e-003
5.0820000e+001 -2.8551599e-003
5.0840000e+001 5.9142600e-003
5.0860000e+001 1.3256100e-002
5.0880000e+001 2.2535370e-002
5.0900000e+001 1.8864449e-002
5.0920000e+001 9.5851793e-003
5.0940000e+001 9.2792701e-003
5.0960000e+001 7.8516895e-003
5.0980000e+001 9.1772997e-003
5.1000000e+001 1.6315200e-003
5.1020000e+001 -8.5654793e-003
5.1040000e+001 -2.0393999e-002
5.1060000e+001 -1.5091560e-002
5.1080000e+001 -7.1378996e-004
5.1100000e+001 1.0910789e-002
5.1120000e+001 1.0095029e-002
5.1140000e+001 4.8945600e-003
5.1160000e+001 3.0591000e-004
5.1180000e+001 -7.0359299e-003
5.1200000e+001 -1.0910789e-002
5.1220000e+001 -1.5295499e-003
5.1240000e+001 7.0359299e-003
5.1260000e+001 1.4071860e-002
5.1280000e+001 6.8319896e-003
5.1300000e+001 2.0393999e-003
5.1320000e+001 3.5689499e-003
5.1340000e+001 9.1772997e-003
5.1360000e+001 8.7694201e-003
5.1380000e+001 5.3024396e-003
5.1400000e+001 2.4472800e-003
5.1420000e+001 1.6315200e-003
5.1440000e+001 1.3256099e-003
5.1460000e+001 -1.1216700e-003
5.1480000e+001 -3.4669800e-003
5.1500000e+001 -6.8319896e-003
5.1520000e+001 -5.5063799e-003
5.1540000e+001 -2.0393999e-003
5.1560000e+001 1.4275799e-003
5.1580000e+001 4.8945600e-003
5.1600000e+001 9.8910895e-003
5.1620000e+001 5.9142600e-003
5.1640000e+001 -4.8945600e-003
5.1660000e+001 -1.2134429e-002
5.1680000e+001 -9.6871497e-003
5.1700000e+001 -6.1181998e-003
5.1720000e+001 -2.4472800e-003
5.1740000e+001 3.2630399e-003
5.1760000e+001 3.8748598e-003
5.1780000e+001 1.4275799e-003
5.1800000e+001 -1.0197000e-004
5.1820000e+001 -7.1378996e-004
5.1840000e+001 6.1182000e-004
5.1860000e+001 1.0197000e-003
5.1880000e+001 2.4472800e-003
5.1900000e+001 1.3256099e-003
5.1920000e+001 4.0787999e-004
5.1940000e+001 -8.1575998e-004
5.1960000e+001 -2.7531900e-003
5.1980000e+001 -5.6083498e-003
5.2000000e+001 -8.5654793e-003
5.2020000e+001 -9.4832099e-003
5.2040000e+001 -6.2201697e-003
5.2060000e+001 -3.6709198e-003
5.2080000e+001 -2.0394000e-004
5.2100000e+001 3.3650098e-003
5.2120000e+001 3.7728899e-003
5.2140000e+001 -1.4275799e-003
5.2160000e+001 -4.5886498e-003
5.2180000e+001 -1.0502910e-002
5.2200000e+001 -7.8516895e-003
5.2220000e+001 -2.3453099e-003
5.2240000e+001 3.1610698e-003
5.2260000e+001 4.1807698e-003
5.2280000e+001 2.6512198e-003
5.2300000e+001 1.3256099e-003
5.2320000e+001 1.5295499e-003
5.2340000e+001 6.2201697e-003
5.2360000e+001 7.4438099e-003
5.2380000e+001 7.5457798e-003
5.2400000e+001 7.7497196e-003
5.2420000e+001 7.4438099e-003
5.2440000e+001 7.2398697e-003
5.2460000e+001 5.8122896e-003
5.2480000e+001 3.8748598e-003
5.2500000e+001 2.1413698e-003
5.2520000e+001 -2.0394000e-004
5.2540000e+001 -3.3650098e-003
5.2560000e+001 -6.5260799e-003
5.2580000e+001 -6.9339600e-003
5.2600000e+001 -5.2004697e-003
5.2620000e+001 -5.5063799e-003
5.2640000e+001 -5.5063799e-003
5.2660000e+001 -5.8122896e-003
5.2680000e+001 -3.3650098e-003
5.2700000e+001 0.0000000e+000
5.2720000e+001 3.3650098e-003
5.2740000e+001 7.1378998e-003
5.2760000e+001 9.7891200e-003
5.2780000e+001 8.6674497e-003
5.2800000e+001 8.1575997e-003
5.2820000e+001 6.9339600e-003
5.2840000e+001 7.6477497e-003
5.2860000e+001 7.8516895e-003
5.2880000e+001 8.6674497e-003
5.2900000e+001 6.2201697e-003
5.2920000e+001 3.7728899e-003
5.2940000e+001 7.1378996e-004
5.2960000e+001 -1.0197000e-004
5.2980000e+001 4.0787999e-004
5.3000000e+001 5.0984998e-004
5.3020000e+001 -1.0197000e-003
5.3040000e+001 -2.9571300e-003
5.3060000e+001 -4.6906197e-003
5.3080000e+001 -7.0359299e-003
5.3100000e+001 -6.6280498e-003
5.3120000e+001 -4.4866799e-003
5.3140000e+001 -2.8551599e-003
5.3160000e+001 -7.1378996e-004
5.3180000e+001 -2.3453099e-003
5.3200000e+001 -3.3650098e-003
5.3220000e+001 -5.2004697e-003
5.3240000e+001 -6.1181998e-003
5.3260000e+001 -5.7103197e-003
5.3280000e+001 -5.5063799e-003
5.3300000e+001 -5.0984998e-003
5.3320000e+001 -6.7300197e-003
5.3340000e+001 -9.2792701e-003
5.3360000e+001 -1.1624579e-002
5.3380000e+001 -1.4275800e-002
5.3400000e+001 -1.4071860e-002
5.3420000e+001 -1.4275800e-002
5.3440000e+001 -1.2338370e-002
5.3460000e+001 -9.7891200e-003
5.3480000e+001 -7.5457798e-003
5.3500000e+001 -4.6906197e-003
5.3520000e+001 -3.4669800e-003
5.3540000e+001 -4.2827397e-003
5.3560000e+001 -4.5886498e-003
5.3580000e+001 -5.6083498e-003
5.3600000e+001 -6.4241100e-003
5.3620000e+001 -7.3418396e-003
5.3640000e+001 -8.1575997e-003
5.3660000e+001 -7.4438099e-003
5.3680000e+001 -5.3024396e-003
5.3700000e+001 -3.7728899e-003
5.3720000e+001 -2.6512198e-003
5.3740000e+001 -1.4275799e-003

In [ ]:
%%writefile project/code/numerics.py
"""
Numerical validation for:
Machine Learning Based Optimal Control of Delayed Sobolev-Type Fractional Evolution
Systems of Order (1,2) under Seismic Disturbances.

Model (1D viscoelastic structure, w in (0,1), Dirichlet ends):
  cD^rho_t (I - a d_ww) xi = c^2 d_ww xi - zeta*tanh(xi(t-tau)) + b(w) mu(t-kappa) - m(w) varpi(t)
  xi(theta) = phi(theta) + g(xi),  theta in [-tau,0],  g(xi) = c_g E^{-1} xi(t_1),  xi_t(0) = 0.
Spectral Galerkin in space (sine modes), Diethelm-Ford-Freed Adams-Bashforth-Moulton in time.
"""
import json, math, time, os
import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

torch.set_default_dtype(torch.float64)
torch.manual_seed(0); np.random.seed(0)
OUT = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "results") + os.sep
import os; os.makedirs(OUT, exist_ok=True)

# ---------------- parameters ----------------
P = dict(rho=1.6, a=0.1, c=1.0, zeta=0.1, c_g=0.1, t1=2.0, tau=0.2, kappa=0.1,
         Theta=4.0, h=0.02, N=8, Nw=64, A0=0.05, rho_u=1e-4, umax=10.0, eqscale=4.0)

def setup(p):
    N, Nw = p["N"], p["Nw"]
    n = np.arange(1, N + 1)
    lam = (n * np.pi) ** 2
    Ediag = 1 + p["a"] * lam                     # eigenvalues of E
    om2 = p["c"] ** 2 * lam / Ediag              # -eigenvalues of G E^{-1}
    w = (np.arange(Nw) + 0.5) / Nw               # midpoint grid
    Phi = np.sqrt(2) * np.sin(np.outer(w, n * np.pi))   # (Nw,N) basis values
    Wq = Phi.T / Nw                               # projection: coeffs = Wq @ values
    b = np.exp(-((w - 0.5) / 0.1) ** 2)           # actuator profile
    m = np.ones(Nw)                               # uniform inertial load
    return dict(n=n, lam=lam, E=Ediag, om2=om2, w=w, Phi=Phi, Wq=Wq,
                b_n=Wq @ b, m_n=Wq @ m, b=b, m=m)

def weights(M, h, al):
    """DFF predictor B and corrector A weights for order al (1<al<2)."""
    k = np.arange(M + 1)
    B = np.zeros((M + 1, M + 1)); A = np.zeros((M + 1, M + 1)); Adiag = np.zeros(M + 1)
    c = h ** al / (al * (al + 1))
    for kk in range(M):  # step kk -> kk+1
        j = np.arange(kk + 1)
        B[kk + 1, :kk + 1] = h ** al / al * ((kk + 1 - j) ** al - (kk - j) ** al)
        a = np.zeros(kk + 1)
        a[0] = kk ** (al + 1) - (kk - al) * (kk + 1) ** al
        jj = np.arange(1, kk + 1)
        a[1:] = (kk - jj + 2) ** (al + 1) + (kk - jj) ** (al + 1) - 2 * (kk - jj + 1) ** (al + 1)
        A[kk + 1, :kk + 1] = c * a
        Adiag[kk + 1] = c
    return B / math.gamma(al), A / math.gamma(al), Adiag / math.gamma(al)

class Solver:
    def __init__(self, p):
        self.p = p; self.S = setup(p)
        self.M = int(round(p["Theta"] / p["h"])); self.t = np.arange(self.M + 1) * p["h"]
        B, A, Ad = weights(self.M, p["h"], p["rho"])
        T = lambda x: torch.tensor(x)
        self.B, self.A, self.Ad = T(B), T(A), T(Ad)
        S = self.S
        self.E, self.om2 = T(S["E"]), T(S["om2"])
        self.Phi, self.Wq = T(S["Phi"]), T(S["Wq"])
        self.b_n, self.m_n = T(S["b_n"]), T(S["m_n"])
        self.dtau = int(round(p["tau"] / p["h"])); self.dkap = int(round(p["kappa"] / p["h"]))
        self.i1 = int(round(p["t1"] / p["h"]))
        phi = p["A0"] * np.sin(np.pi * S["w"])        # constant history shape
        self.phi_n = T(S["Wq"] @ phi)

    def rhs(self, y, ydel, u, v):
        # modal right-hand side of D^rho y = -om2 y + F_n / E_n
        xdel = self.Phi @ ydel
        J = -self.p["zeta"] * torch.tanh(xdel)
        F = self.Wq @ J + self.b_n * u - self.m_n * v
        return -self.om2 * y + F / self.E

    def solve_given_g(self, g_n, u, v):
        """u,v: tensors of length M+1 (u already delayed-consistent: u[k]=mu(t_k-kappa))."""
        y0 = self.phi_n + g_n
        Y = [y0]; Fs = [self.rhs(y0, y0, u[0], v[0])]
        for k in range(self.M):
            Fst = torch.stack(Fs)
            kd = k + 1 - self.dtau
            ydel = Y[kd] if kd >= 0 else y0
            pred = y0 + self.B[k + 1, :k + 1] @ Fst
            fpred = self.rhs(pred, ydel, u[k + 1], v[k + 1])
            corr = y0 + self.A[k + 1, :k + 1] @ Fst + self.Ad[k + 1] * fpred
            Y.append(corr); Fs.append(self.rhs(corr, ydel, u[k + 1], v[k + 1]))
        return torch.stack(Y)          # (M+1, N)

    def solve(self, mu, varpi, psi=None, iters=30, tol=1e-12):
        """mu: control on grid t_k (length M+1); delayed by kappa using history psi (default 0)."""
        dk = self.dkap
        hist = torch.zeros(dk) if psi is None else psi
        u = torch.cat([hist, mu[:self.M + 1 - dk]]) if dk > 0 else mu
        g = torch.zeros_like(self.phi_n)
        for _ in range(iters):
            Y = self.solve_given_g(g, u, varpi)
            g_new = self.p["c_g"] * Y[self.i1] / self.E      # g = c_g E^{-1} xi(t1)
            if torch.max(torch.abs(g_new - g)) < tol: g = g_new; break
            g = g_new.detach() if not g_new.requires_grad else g_new
        Y = self.solve_given_g(g, u, varpi)
        return Y, g

    def l2norm_t(self, Y):  # ||xi(t)||_{L2} = Euclidean norm of modal coefficients
        return torch.sqrt(torch.sum(Y ** 2, dim=1))

    def supnorm(self, Y, g):
        hist = torch.sqrt(torch.sum((self.phi_n + g) ** 2))
        return torch.maximum(self.l2norm_t(Y).max(), hist)

    def cost(self, Y, mu):
        h = self.p["h"]
        return h * torch.sum(self.l2norm_t(Y) ** 2 + self.p["rho_u"] * mu ** 2)

# ---------------- synthetic earthquake (Kanai-Tajimi + envelope) ----------------
def kanai_tajimi(t, seed, wg=3.0, zg=0.4, scale=1.0):
    """Filtered white noise ground acceleration with a Jennings-type envelope.
    Time t is non-dimensional; wg, zg are ground frequency and damping in these units."""
    rng = np.random.default_rng(seed)
    dt = t[1] - t[0]; nsub = 20; dts = dt / nsub
    n = len(t) * nsub
    wn = rng.standard_normal(n) / np.sqrt(dts)
    x = v = 0.0; acc = np.zeros(n)
    for i in range(n):
        a_rel = -2 * zg * wg * v - wg ** 2 * x - wn[i]
        v += a_rel * dts; x += v * dts
        acc[i] = -(2 * zg * wg * v + wg ** 2 * x)
    acc = acc[::nsub][:len(t)]
    T = t[-1]; t1e, t2e = 0.1 * T, 0.45 * T
    env = np.where(t < t1e, (t / t1e) ** 2, np.where(t < t2e, 1.0, np.exp(-2.5 * (t - t2e) / T)))
    g = acc * env
    return scale * g / np.max(np.abs(g))

def l2t(x, h):  # L2(0,Theta) norm of a grid function (trapezoid)
    x = np.asarray(x); return math.sqrt(h * (np.sum(x ** 2) - 0.5 * (x[0] ** 2 + x[-1] ** 2)))

# ---------------- theoretical constants ----------------
def theory_constants(p, S):
    vs = p["rho"] / 2
    Einv = 1 / S["E"].min()
    Ups = 1.0
    UE = Ups * Einv; UP = UE / math.gamma(2 * vs)
    fac = 2 * UE * p["Theta"] ** (2 * vs) / math.gamma(2 * vs + 1)
    lamJ = etaJ = p["zeta"]; lamg = eta1 = abs(p["c_g"])
    g1 = UE * eta1 + etaJ * fac
    g2 = UE * lamg + lamJ * fac
    Dnorm = math.sqrt(np.mean(S["m"] ** 2))
    Lam_w = UP * Dnorm * p["Theta"] ** (2 * vs - 0.5) / (1 - g2)
    return dict(varsigma=vs, Einv=Einv, UpsE=UE, UpsP=UP, gamma1=g1, gamma2=g2,
                Dnorm=Dnorm, Lambda_varpi=Lam_w)

# ---------------- plotting style ----------------
plt.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
                     "mathtext.fontset": "stix", "font.size": 10, "axes.grid": True,
                     "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False,
                     "legend.frameon": False, "figure.dpi": 150})
C = dict(blue="#005AB4", red="#B3261E", green="#2E7D32", amber="#B8860B", grey="#555555", purple="#6A3D9A")

results = {}
t0 = time.time()
sol = Solver(P); S = sol.S; M = sol.M; tg = sol.t; h = P["h"]
TC = theory_constants(P, S); results["theory"] = TC
print("theory", TC)

# ================= Example 1: smooth disturbance, ABM vs fPINN =================
v_s = torch.tensor(0.5 * np.sin(2.0 * tg))
mu0 = torch.zeros(M + 1)
with torch.no_grad():
    Yref, gref = sol.solve(mu0, v_s)
# convergence check of ABM: halve the step
P_f = dict(P); P_f["h"] = P["h"] / 2
sol_f = Solver(P_f)
with torch.no_grad():
    Yf, _ = sol_f.solve(torch.zeros(sol_f.M + 1), torch.tensor(0.5 * np.sin(2.0 * sol_f.t)))
err_h = float(torch.max(torch.abs(Yf[::2] - Yref)))
results["abm_step_halving_error"] = err_h

# fPINN in integral form (hard initial constraints, residual of the Volterra equation,
# Fourier-feature input layer, Adam followed by L-BFGS)
class Net(torch.nn.Module):
    def __init__(s, N, width=64, nf=8):
        super().__init__()
        s.k = torch.arange(1, nf + 1).double() * 0.8
        s.f = torch.nn.Sequential(torch.nn.Linear(2 * nf + 1, width), torch.nn.Tanh(),
                                  torch.nn.Linear(width, width), torch.nn.Tanh(),
                                  torch.nn.Linear(width, width), torch.nn.Tanh(),
                                  torch.nn.Linear(width, N))
    def forward(s, t):
        z = t * P["Theta"]
        return s.f(torch.cat([t, torch.sin(s.k * z), torch.cos(s.k * z)], 1))

torch.manual_seed(1)
tt = torch.tensor(tg).unsqueeze(1)
net = Net(P["N"]); g_par = torch.zeros(P["N"], requires_grad=True)
Aful = sol.A.clone(); Aful[torch.arange(M + 1), torch.arange(M + 1)] = sol.Ad  # product-trapezoidal weights
SC = 1e4
def pinn_loss():
    y0 = sol.phi_n + g_par
    Y = y0 + (tt / P["Theta"]) ** 2 * net(tt / P["Theta"])      # y(0)=y0 and y'(0)=0 built in
    idx = torch.arange(M + 1) - sol.dtau
    Ydel = torch.where((idx >= 0).unsqueeze(1), Y[idx.clamp(min=0)], y0)
    Jn = (-P["zeta"] * torch.tanh(Ydel @ sol.Phi.T)) @ sol.Wq.T
    F = -sol.om2 * Y + (Jn - sol.m_n * v_s.unsqueeze(1)) / sol.E
    R = Y - y0 - Aful @ F
    return torch.mean(R ** 2) + torch.sum((g_par - P["c_g"] * Y[sol.i1] / sol.E) ** 2), Y
opt = torch.optim.Adam(list(net.parameters()) + [g_par], lr=2e-3)
hist_loss = []
for it in range(20000):
    opt.zero_grad(); l, _ = pinn_loss(); (SC * l).backward(); opt.step()
    if it == 14000:
        for gr in opt.param_groups: gr["lr"] = 5e-4
    if it % 100 == 0: hist_loss.append((it, float(l.detach())))
optL = torch.optim.LBFGS(list(net.parameters()) + [g_par], max_iter=2000, tolerance_grad=1e-12,
                         tolerance_change=1e-15, history_size=50, line_search_fn="strong_wolfe")
def closure():
    optL.zero_grad(); l, _ = pinn_loss(); (SC * l).backward(); return SC * l
for k in range(6):
    optL.step(closure)
    with torch.no_grad(): hist_loss.append((20000 + 2000 * (k + 1), float(pinn_loss()[0])))
with torch.no_grad():
    lval, Ypinn = pinn_loss()
xref = (Yref @ sol.Phi.T).numpy(); xpinn = (Ypinn @ sol.Phi.T).numpy()
err_abs = np.abs(xref - xpinn)
rel_l2 = float(np.linalg.norm(xref - xpinn) / np.linalg.norm(xref))
results["fpinn"] = dict(max_abs=float(err_abs.max()), rel_l2=rel_l2, final_loss=float(lval), peak=float(np.abs(xref).max()),
                        g_err=float(torch.max(torch.abs(g_par - gref))))
print("fPINN", results["fpinn"], "time", time.time() - t0)

# Figure 1: surface of reference + error
fig = plt.figure(figsize=(7.2, 2.9))
ax = fig.add_subplot(1, 2, 1, projection="3d")
WW, TT = np.meshgrid(S["w"], tg)
ax.plot_surface(TT, WW, xref, cmap="viridis", linewidth=0, antialiased=True)
ax.set_xlabel(r"$t$"); ax.set_ylabel(r"$w$"); ax.set_zlabel(r"$\xi(t,w)$"); ax.set_title("(a) ABM reference solution", fontsize=9)
ax.view_init(28, -125)
ax2 = fig.add_subplot(1, 2, 2)
cs = ax2.contourf(TT, WW, err_abs, 30, cmap="magma"); fig.colorbar(cs, ax=ax2, fraction=0.046)
ax2.set_xlabel(r"$t$"); ax2.set_ylabel(r"$w$"); ax2.set_title(r"(b) $|\xi_{\mathrm{ABM}}-\xi_{\mathrm{fPINN}}|$", fontsize=9)
ax2.grid(False)
fig.tight_layout(); fig.savefig(OUT + "fig1_solution_error.pdf"); fig.savefig(OUT + "fig1_solution_error.png"); plt.close(fig)

# Figure 2: midpoint comparison + training loss
iw = np.argmin(np.abs(S["w"] - 0.5))
fig, axs = plt.subplots(1, 2, figsize=(7.2, 2.7))
axs[0].plot(tg, xref[:, iw], color=C["blue"], lw=1.6, label="ABM (classical)")
axs[0].plot(tg[::6], xpinn[::6, iw], "o", ms=3.2, mfc="none", color=C["red"], label="fPINN")
axs[0].set_xlabel(r"$t$"); axs[0].set_ylabel(r"$\xi(t,1/2)$"); axs[0].legend(); axs[0].set_title("(a) midpoint displacement", fontsize=9)
hl = np.array(hist_loss)
axs[1].semilogy(hl[:, 0], hl[:, 1], color=C["green"], lw=1.4)
axs[1].set_xlabel("iteration"); axs[1].set_ylabel("loss"); axs[1].set_title("(b) fPINN training loss", fontsize=9)
fig.tight_layout(); fig.savefig(OUT + "fig2_fpinn.pdf"); fig.savefig(OUT + "fig2_fpinn.png"); plt.close(fig)

# ================= Effect of the fractional order =================
fig, ax = plt.subplots(figsize=(3.6, 2.7))
ord_res = {}
for rho_, col in zip([1.2, 1.5, 1.8], [C["amber"], C["blue"], C["red"]]):
    Pr = dict(P); Pr["rho"] = rho_; sr = Solver(Pr)
    with torch.no_grad():
        Yr, _ = sr.solve(torch.zeros(sr.M + 1), torch.tensor(0.5 * np.sin(2.0 * sr.t)))
    xm = (Yr @ sr.Phi.T).numpy()[:, iw]
    ax.plot(sr.t, xm, color=col, lw=1.4, label=rf"$\varrho={rho_}$")
    ord_res[str(rho_)] = float(np.max(np.abs(xm)))
ax.set_xlabel(r"$t$"); ax.set_ylabel(r"$\xi(t,1/2)$"); ax.legend()
fig.tight_layout(); fig.savefig(OUT + "fig3_order.pdf"); fig.savefig(OUT + "fig3_order.png"); plt.close(fig)
results["order_peak"] = ord_res

# ================= Example 2: seismic disturbance =================
eq1 = kanai_tajimi(tg, seed=11, scale=P["eqscale"])
eq2 = kanai_tajimi(tg, seed=29, scale=P["eqscale"])
v1 = torch.tensor(eq1)

# --- robustness test: perturbations of eq1 ---
rob = []
with torch.no_grad():
    Yb, gb = sol.solve(mu0, v1)
    for eps in [0.02, 0.05, 0.1, 0.2, 0.4, 0.8]:
        vp = torch.tensor(eq1 + eps * (eq2 - eq1))
        Yp, gp = sol.solve(mu0, vp)
        dstate = float(torch.max(torch.cat([sol.l2norm_t(Yp - Yb), torch.sqrt(torch.sum((gp - gb) ** 2)).unsqueeze(0)])))
        dw = l2t(eps * (eq2 - eq1), h)
        rob.append((eps, dw, dstate))
rob = np.array(rob)
emp_gain = float(np.max(rob[:, 2] / rob[:, 1]))
results["robustness"] = dict(rows=rob.tolist(), empirical_gain=emp_gain, theoretical_gain=TC["Lambda_varpi"])

# --- optimal control: NN controller vs classical direct method ---
def train_nn(solver, vdist, iters=500, lr=1e-2, seed=0):
    torch.manual_seed(seed)
    ctrl = torch.nn.Sequential(torch.nn.Linear(1, 32), torch.nn.Tanh(), torch.nn.Linear(32, 32),
                               torch.nn.Tanh(), torch.nn.Linear(32, 1))
    tt_ = torch.tensor(solver.t).unsqueeze(1) / solver.p["Theta"]
    opt_ = torch.optim.Adam(ctrl.parameters(), lr=lr); hist = []
    for it in range(iters):
        opt_.zero_grad()
        mu = P["umax"] * torch.tanh(ctrl(tt_).squeeze(1))          # |mu| <= 3 : admissible set
        Y, g = solver.solve(mu, vdist, iters=2)
        Jc = solver.cost(Y, mu); Jc.backward(); opt_.step(); hist.append(float(Jc))
    with torch.no_grad():
        mu = P["umax"] * torch.tanh(ctrl(tt_).squeeze(1)); Y, g = solver.solve(mu, vdist)
        return mu, Y, g, float(solver.cost(Y, mu)), hist

def train_direct(solver, vdist, iters=500, lr=5e-2):
    z = torch.zeros(solver.M + 1, requires_grad=True)
    opt_ = torch.optim.Adam([z], lr=lr); hist = []
    for it in range(iters):
        opt_.zero_grad()
        mu = P["umax"] * torch.tanh(z)
        Y, g = solver.solve(mu, vdist, iters=2)
        Jc = solver.cost(Y, mu); Jc.backward(); opt_.step(); hist.append(float(Jc))
    with torch.no_grad():
        mu = P["umax"] * torch.tanh(z); Y, g = solver.solve(mu, vdist)
        return mu, Y, g, float(solver.cost(Y, mu)), hist

with torch.no_grad():
    J_unc = float(sol.cost(Yb, mu0))
t1_ = time.time()
mu_nn, Y_nn, g_nn, J_nn, h_nn = train_nn(sol, v1)
t_nn = time.time() - t1_; t1_ = time.time()
mu_d, Y_d, g_d, J_d, h_d = train_direct(sol, v1)
t_d = time.time() - t1_
results["oc"] = dict(J_uncontrolled=J_unc, J_nn=J_nn, J_direct=J_d, time_nn=t_nn, time_direct=t_d,
                     peak_unc=float((Yb @ sol.Phi.T).abs().max()), peak_nn=float((Y_nn @ sol.Phi.T).abs().max()),
                     peak_direct=float((Y_d @ sol.Phi.T).abs().max()),
                     rms_unc=l2t(sol.l2norm_t(Yb).numpy(), h) / math.sqrt(P["Theta"]),
                     rms_nn=l2t(sol.l2norm_t(Y_nn).numpy(), h) / math.sqrt(P["Theta"]),
                     rms_direct=l2t(sol.l2norm_t(Y_d).numpy(), h) / math.sqrt(P["Theta"]))
print("OC", results["oc"], time.time() - t0)

# Figure 4: earthquake + controlled/uncontrolled midpoint + control
fig, axs = plt.subplots(3, 1, figsize=(7.0, 5.6), sharex=True)
axs[0].plot(tg, eq1, color=C["grey"], lw=0.9); axs[0].set_ylabel(r"$\varpi(t)$"); axs[0].set_title("(a) synthetic ground acceleration (record 1)", fontsize=9)
xb = (Yb @ sol.Phi.T).numpy()[:, iw]; xn = (Y_nn @ sol.Phi.T).numpy()[:, iw]; xd = (Y_d @ sol.Phi.T).numpy()[:, iw]
axs[1].plot(tg, xb, color=C["red"], lw=1.3, label="uncontrolled")
axs[1].plot(tg, xd, color=C["amber"], lw=1.3, ls="--", label="direct method")
axs[1].plot(tg, xn, color=C["blue"], lw=1.5, label="NN open-loop solver")
axs[1].set_ylabel(r"$\xi(t,1/2)$"); axs[1].legend(ncol=3, fontsize=8); axs[1].set_title("(b) midpoint displacement", fontsize=9)
axs[2].plot(tg, mu_d.numpy(), color=C["amber"], lw=1.2, ls="--", label="direct method")
axs[2].plot(tg, mu_nn.numpy(), color=C["blue"], lw=1.5, label="NN open-loop solver")
axs[2].axvspan(0, P["kappa"], color=C["grey"], alpha=0.15, lw=0)
axs[2].set_ylabel(r"$\mu(t)$"); axs[2].set_xlabel(r"$t$"); axs[2].legend(fontsize=8); axs[2].set_title(r"(c) optimal control (shaded: $[0,\kappa]$)", fontsize=9)
fig.tight_layout(); fig.savefig(OUT + "fig4_seismic_control.pdf"); fig.savefig(OUT + "fig4_seismic_control.png"); plt.close(fig)

# Figure 5: training histories + robustness
fig, axs = plt.subplots(1, 2, figsize=(7.2, 2.8))
axs[0].semilogy(h_nn, color=C["blue"], lw=1.4, label="NN open-loop solver")
axs[0].semilogy(h_d, color=C["amber"], lw=1.4, ls="--", label="direct method")
axs[0].axhline(J_unc, color=C["red"], lw=1.0, ls=":", label="uncontrolled")
axs[0].set_xlabel("iteration"); axs[0].set_ylabel(r"$\mathcal{J}(\mu)$"); axs[0].legend(fontsize=8); axs[0].set_title("(a) cost during training", fontsize=9)
axs[1].loglog(rob[:, 1], rob[:, 2], "o-", color=C["blue"], ms=4, lw=1.3, label=r"computed $\|\xi^{\varpi^1}-\xi^{\varpi^2}\|_\ast$")
xx = np.array([rob[:, 1].min(), rob[:, 1].max()])
axs[1].loglog(xx, TC["Lambda_varpi"] * xx, color=C["red"], lw=1.3, ls="--", label=r"bound $\Lambda_\varpi\|\varpi^1-\varpi^2\|$")
axs[1].set_xlabel(r"$\|\varpi^1-\varpi^2\|_{L^2}$"); axs[1].legend(fontsize=8); axs[1].set_title("(b) robustness estimate (4.6)", fontsize=9)
fig.tight_layout(); fig.savefig(OUT + "fig5_training_robustness.pdf"); fig.savefig(OUT + "fig5_training_robustness.png"); plt.close(fig)

# ================= Effect of the control delay =================
kap_res = {}
for kap in [0.0, 0.1, 0.3]:
    Pk = dict(P); Pk["kappa"] = kap; sk = Solver(Pk)
    _, Yk, _, Jk, _ = train_nn(sk, v1, iters=500)
    kap_res[str(kap)] = dict(J=Jk, peak=float((Yk @ sk.Phi.T).abs().max()))
results["delay"] = kap_res
print("delay", kap_res, time.time() - t0)

# ================= Robustness of the optimal cost (Corollary 5.4) =================
v2 = torch.tensor(eq1 + 0.2 * (eq2 - eq1))
_, Y2, _, J2, _ = train_nn(sol, v2, iters=500)
with torch.no_grad():
    Rmax = float(max(sol.supnorm(Yb, gb), sol.supnorm(Y2, torch.zeros(P["N"]))))
results["value"] = dict(V1=J_nn, V2=J2, diff=abs(J_nn - J2), dw=l2t(0.2 * (eq2 - eq1), h))
results["time_total"] = time.time() - t0
json.dump(results, open(OUT + "results.json", "w"), indent=2)
print(json.dumps(results, indent=2))


In [ ]:
%%writefile project/code/fig0_model.py
"""Figure: the model problem (schematic, spatial profiles, natural frequencies)."""
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Rectangle
plt.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
                     "mathtext.fontset": "stix", "font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False, "legend.frameon": False})
a, c, A0 = 0.1, 1.0, 0.05
w = np.linspace(0, 1, 400)
b = np.exp(-((w - 0.5) / 0.1) ** 2)
phi = A0 * np.sin(np.pi * w)
n = np.arange(1, 9); lam = (n * np.pi) ** 2; om = np.sqrt(c ** 2 * lam / (1 + a * lam))
BLUE, AMBER, GREY, RED = "#2a788e", "#e8a202", "#555555", "#b3261e"

fig = plt.figure(figsize=(7.2, 4.9))
gs = fig.add_gridspec(2, 2, height_ratios=[0.95, 1], hspace=0.38, wspace=0.32)

# (a) schematic
ax = fig.add_subplot(gs[0, :]); ax.set_xlim(-0.12, 1.12); ax.set_ylim(-0.56, 0.62); ax.axis("off")
ax.fill_between([-0.12, 1.12], -0.55, -0.42, color="#d9d9d9", lw=0)
ax.plot([-0.12, 1.12], [-0.42, -0.42], color=GREY, lw=1)
for x0 in [-0.04, 1.0]:
    ax.add_patch(Rectangle((x0, -0.42), 0.04, 0.42, color=GREY))
defl = 0.22 * np.sin(np.pi * w)
ax.plot(w, 0 * w, color="#bbbbbb", lw=1.2, ls="--")
ax.plot(w, defl, color=BLUE, lw=3)
ax.annotate(r"$\xi(t,w)$", (0.25, 0.22 * np.sin(np.pi * 0.25)), xytext=(0.12, 0.42),
            arrowprops=dict(arrowstyle="-", color=GREY, lw=0.8), fontsize=10, color=BLUE)
ax.add_patch(FancyArrowPatch((0.5, 0.58), (0.5, 0.26), arrowstyle="-|>", mutation_scale=14, color=AMBER, lw=2))
ax.text(0.53, 0.45, r"actuator $b(w)\,\mu(t-\kappa)$", color="#8a5d00", fontsize=10)
ax.add_patch(FancyArrowPatch((0.22, -0.49), (0.40, -0.49), arrowstyle="<|-|>", mutation_scale=12, color=RED, lw=1.6))
ax.text(0.43, -0.52, r"ground acceleration $\varpi(t)$", color=RED, fontsize=10)
ax.text(-0.11, 0.08, r"$\xi(t,0)=0$", fontsize=9, color=GREY)
ax.text(0.95, 0.08, r"$\xi(t,1)=0$", fontsize=9, color=GREY)
ax.text(0.72, 0.28, r"$-\zeta\tanh\,\xi(t-\tau,w)$", fontsize=9, color=GREY)
ax.set_title(r"(a) viscoelastic structure on $w\in(0,1)$ with fixed ends", fontsize=9)

# (b) spatial profiles
ax2 = fig.add_subplot(gs[1, 0])
ax2.plot(w, b, color=AMBER, lw=2, label=r"actuator profile $b(w)$")
ax2.plot(w, phi / A0, color=BLUE, lw=2, ls="--", label=r"history shape $\phi(w)/A_0$")
ax2.set_xlabel(r"$w$"); ax2.set_ylabel("profile"); ax2.set_ylim(0, 1.6); ax2.grid(alpha=0.3)
ax2.legend(fontsize=8, loc="upper center", ncol=1); ax2.set_title("(b) spatial profiles", fontsize=9)

# (c) natural frequencies
ax3 = fig.add_subplot(gs[1, 1])
ax3.plot(n, om, "o-", color=BLUE, lw=1.8, ms=6, mec="white", mew=1.0, label=r"$\omega_n=\sqrt{c^2\lambda_n/(1+a\lambda_n)}$")
ax3.axhline(c / np.sqrt(a), color=GREY, lw=1, ls=":", label=r"limit $c/\sqrt{a}=3.16$")
ax3.annotate(f"$\\omega_1={om[0]:.2f}$", (1, om[0]), xytext=(1.6, 1.35), fontsize=8,
             arrowprops=dict(arrowstyle="-", color=GREY, lw=0.8))
ax3.set_xlabel(r"mode $n$"); ax3.set_ylabel(r"$\omega_n$"); ax3.set_ylim(0, 3.6); ax3.grid(alpha=0.3)
ax3.legend(fontsize=8, loc="lower right"); ax3.set_title("(c) natural frequencies", fontsize=9)

import os; out = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "results") + os.sep
fig.savefig(out + "fig0_model.pdf", bbox_inches="tight"); fig.savefig(out + "fig0_model.png", bbox_inches="tight", dpi=200)
print(om[:3])


In [ ]:
%%writefile project/code/fig3_order.py
"""Figure 3: effect of the fractional order rho on the response (no control, varpi = 0.5 sin 2t)."""
import numpy as np, torch, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import os
src = open(os.path.join(os.path.dirname(os.path.abspath(__file__)), "numerics.py")).read()
exec(src[:src.index("results = {}")])          # parameters, Solver, plotting style

rhos_fine = np.round(np.arange(1.10, 1.901, 0.05), 2)
rhos_show = [1.2, 1.4, 1.6, 1.8]
curves, peaks, energy = {}, [], []
for r in rhos_fine:
    Pr = dict(P); Pr["rho"] = float(r); s = Solver(Pr)
    with torch.no_grad():
        Y, _ = s.solve(torch.zeros(s.M + 1), torch.tensor(0.5 * np.sin(2.0 * s.t)))
    x = (Y @ s.Phi.T).numpy()
    iw = np.argmin(np.abs(s.S["w"] - 0.5))
    curves[float(r)] = x[:, iw]
    peaks.append(np.abs(x[:, iw]).max())
    nrm = np.sqrt(np.sum(Y.numpy() ** 2, axis=1))
    energy.append(np.sqrt(s.p["h"] * np.sum(nrm ** 2)))
t = s.t
# data behind Figure 3 (CSV)
import csv, os
_d = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "data", "processed"); os.makedirs(_d, exist_ok=True)
with open(os.path.join(_d, "table_order_peaks.csv"), "w", newline="") as f:
    wr = csv.writer(f); wr.writerow(["rho", "peak_midpoint_displacement", "L2_energy_norm"])
    for r, pk, en in zip(rhos_fine, peaks, energy): wr.writerow([f"{r:.2f}", f"{pk:.10g}", f"{en:.10g}"])
with open(os.path.join(_d, "order_midpoint_curves.csv"), "w", newline="") as f:
    wr = csv.writer(f); wr.writerow(["t"] + [f"xi_mid_rho_{r:.2f}" for r in rhos_fine])
    for k in range(len(t)): wr.writerow([f"{t[k]:.4f}"] + [f"{curves[float(r)][k]:.10g}" for r in rhos_fine])

cols = ["#440154", "#2a788e", "#22a884", "#e8a202"]    # viridis-like, dark purple -> amber = low -> high order
styles = ["-", "--", "-.", (0, (1, 1.2))]
fig = plt.figure(figsize=(7.2, 5.4))
gs = fig.add_gridspec(2, 2, height_ratios=[1.05, 1], hspace=0.45, wspace=0.32)

ax = fig.add_subplot(gs[0, :])
for r, col, ls in zip(rhos_show, cols, styles):
    ax.plot(t, curves[r], color=col, lw=2.0, ls=ls, label=rf"$\varrho={r}$")
    k = np.argmax(np.abs(curves[r]))
    ax.plot(t[k], curves[r][k], "o", ms=6, color=col, mec="white", mew=1.0, zorder=3)
ax.axhline(0, color="#999999", lw=0.8)
ax.set_xlim(0, 4); ax.set_xlabel(r"$t$"); ax.set_ylabel(r"$\xi(t,1/2)$")
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, 1.16), fontsize=9, handlelength=3.0)
ax.set_title("(a) midpoint displacement for four orders (dots: peak values)", fontsize=9, pad=24)

ax2 = fig.add_subplot(gs[1, 0])
ax2.plot(rhos_fine, peaks, "-", color="#2a788e", lw=2.0)
ax2.plot(rhos_fine, peaks, "o", ms=5, color="#2a788e", mec="white", mew=1.0)
for r, col in zip(rhos_show, cols):
    i = int(np.argmin(np.abs(rhos_fine - r)))
    ax2.plot(rhos_fine[i], peaks[i], "o", ms=8, color=col, mec="white", mew=1.2, zorder=3)
for r in rhos_show:
    i = int(np.argmin(np.abs(rhos_fine - r)))
    ax2.annotate(f"{peaks[i]:.3f}", (rhos_fine[i], peaks[i]), textcoords="offset points",
                 xytext=(-4, 8), ha="right", fontsize=8, color="#333333")
ax2.set_xlabel(r"order $\varrho$"); ax2.set_ylabel(r"peak $|\xi(t,1/2)|$")
ax2.set_title("(b) peak displacement versus order", fontsize=9)

ax3 = fig.add_subplot(gs[1, 1])
Z = np.array([curves[float(r)] for r in rhos_fine])
lim = np.abs(Z).max()
im = ax3.pcolormesh(t, rhos_fine, Z, cmap="RdBu_r", norm=TwoSlopeNorm(0, -lim, lim), shading="auto")
cb = fig.colorbar(im, ax=ax3, fraction=0.05, pad=0.03); cb.set_label(r"$\xi(t,1/2)$", fontsize=9)
for r, col, ls in zip(rhos_show, cols, styles):
    ax3.axhline(r, color=col, lw=1.4, ls=ls)
ax3.set_xlabel(r"$t$"); ax3.set_ylabel(r"order $\varrho$"); ax3.grid(False)
ax3.set_title(r"(c) response map over $(t,\varrho)$", fontsize=9)

import os; out = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "results") + os.sep
fig.savefig(out + "fig3_order.pdf", bbox_inches="tight"); fig.savefig(out + "fig3_order.png", bbox_inches="tight", dpi=200)
print({str(r): round(float(p), 4) for r, p in zip(rhos_fine, peaks)})


In [ ]:
%%writefile project/code/numerics_elcentro.py
"""Seismic part of the numerical study with the recorded 1940 El Centro NS ground acceleration.
Record: data/raw/elcentro_NS.txt (time [s], acceleration [g]), dt = 0.02 s, 2688 points.
Time scaling: the first natural period of the model structure is set to T1 = 1.5 s,
so one model time unit = omega_1 * T1 / (2 pi) seconds. The record is normalized so that
its peak ground acceleration equals 4 model units."""
import json, math, time, os
import numpy as np, torch, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
here = os.path.dirname(os.path.abspath(__file__))
src = open(os.path.join(here, "numerics.py")).read()
exec(src[:src.index("results = {}")])          # parameters, Solver, helpers, plotting style
_i = src.index("def train_nn"); _j = src.index("with torch.no_grad():\n    J_unc")
exec(src[_i:_j])                               # train_nn, train_direct
# Controller network with a Fourier-feature input layer (needed for the long, oscillatory record)
class FNet(torch.nn.Module):
    def __init__(s, nf=12, width=48):
        super().__init__()
        s.k = torch.arange(1, nf + 1).double() * math.pi
        s.f = torch.nn.Sequential(torch.nn.Linear(2 * nf + 1, width), torch.nn.Tanh(),
                                  torch.nn.Linear(width, width), torch.nn.Tanh(), torch.nn.Linear(width, 1))
    def forward(s, t): return s.f(torch.cat([t, torch.sin(s.k * t), torch.cos(s.k * t)], 1))

def train_nn(solver, vdist, iters=400, lr=1e-2, seed=0):
    torch.manual_seed(seed); ctrl = FNet()
    tt_ = torch.tensor(solver.t).unsqueeze(1) / solver.p["Theta"]
    opt_ = torch.optim.Adam(ctrl.parameters(), lr=lr); hist = []
    for it in range(iters):
        opt_.zero_grad(); mu = solver.p["umax"] * torch.tanh(ctrl(tt_).squeeze(1))
        Y, g = solver.solve(mu, vdist, iters=2); Jc = solver.cost(Y, mu); Jc.backward(); opt_.step()
        hist.append(float(Jc.detach()))
    with torch.no_grad():
        mu = solver.p["umax"] * torch.tanh(ctrl(tt_).squeeze(1)); Y, g = solver.solve(mu, vdist)
        return mu, Y, g, float(solver.cost(Y, mu)), hist
NPARAM = sum(p.numel() for p in FNet().parameters())


P2 = dict(P); P2.update(Theta=15.0, h=0.05, zeta=0.01)
T1, PGA_ND, SHIFT = 1.5, 4.0, 1.0            # period [s], normalized PGA, shift [s] of 2nd record
sol2 = Solver(P2); S2 = sol2.S; M2 = sol2.M; tg2 = sol2.t; h2 = P2["h"]
om1 = float(np.sqrt(S2["om2"][0]))
tscale = om1 * T1 / (2 * np.pi)              # seconds per model time unit
rec = np.loadtxt(os.path.join(here, "..", "data", "raw", "elcentro_NS.txt"))
tr, ar = rec[:, 0], rec[:, 1]
pga = np.abs(ar).max()
def record(shift=0.0):
    return PGA_ND / pga * np.interp(tg2 * tscale - shift, tr, ar, left=0.0, right=0.0)
eq1 = record(0.0); eq2 = record(SHIFT)
TC2 = theory_constants(P2, S2)
res = dict(nparam_nn=NPARAM, theory=TC2, tscale=tscale, phys_window=float(tg2[-1] * tscale), pga_g=float(pga), omega1=om1)
print("theory", TC2, "window [s]", res["phys_window"])

v1 = torch.tensor(eq1); mu0 = torch.zeros(M2 + 1)
with torch.no_grad():
    Yb, gb = sol2.solve(mu0, v1)
    J_unc = float(sol2.cost(Yb, mu0))
# robustness
rob = []
with torch.no_grad():
    for eps in [0.02, 0.05, 0.1, 0.2, 0.4, 0.8]:
        vp = torch.tensor(eq1 + eps * (eq2 - eq1)); Yp, gp = sol2.solve(mu0, vp)
        d = float(torch.max(torch.cat([sol2.l2norm_t(Yp - Yb), torch.sqrt(torch.sum((gp - gb) ** 2)).unsqueeze(0)])))
        rob.append((eps, l2t(eps * (eq2 - eq1), h2), d))
rob = np.array(rob)
res["robustness"] = dict(rows=rob.tolist(), empirical_gain=float(np.max(rob[:, 2] / rob[:, 1])), theoretical_gain=TC2["Lambda_varpi"])
print("rob", res["robustness"]["empirical_gain"], time.time())

t0 = time.time()
mu_nn, Y_nn, g_nn, J_nn, h_nn = train_nn(sol2, v1); t_nn = time.time() - t0; t0 = time.time()
mu_d, Y_d, g_d, J_d, h_d = train_direct(sol2, v1); t_d = time.time() - t0
X = lambda Y: (Y @ sol2.Phi.T)
res["oc"] = dict(J_uncontrolled=J_unc, J_nn=J_nn, J_direct=J_d, time_nn=t_nn, time_direct=t_d,
                 peak_unc=float(X(Yb).abs().max()), peak_nn=float(X(Y_nn).abs().max()), peak_direct=float(X(Y_d).abs().max()),
                 rms_unc=l2t(sol2.l2norm_t(Yb).numpy(), h2) / math.sqrt(P2["Theta"]),
                 rms_nn=l2t(sol2.l2norm_t(Y_nn).numpy(), h2) / math.sqrt(P2["Theta"]),
                 rms_direct=l2t(sol2.l2norm_t(Y_d).numpy(), h2) / math.sqrt(P2["Theta"]),
                 umax_nn=float(mu_nn.abs().max()), umax_direct=float(mu_d.abs().max()))
print("OC", res["oc"])

iw = int(np.argmin(np.abs(S2["w"] - 0.5))); ts = tg2 * tscale
fig, axs = plt.subplots(3, 1, figsize=(7.0, 5.8), sharex=True)
axs[0].plot(tr, ar, color="#bbbbbb", lw=0.7)
axs[0].plot(ts, eq1 * pga / PGA_ND, color=C["grey"], lw=1.0)
axs[0].set_xlim(0, ts[-1]); axs[0].set_ylabel(r"$a_g$ [g]")
axs[0].set_title("(a) El Centro 1940, NS component (recorded)", fontsize=9)
xb, xn, xd = X(Yb).numpy()[:, iw], X(Y_nn).numpy()[:, iw], X(Y_d).numpy()[:, iw]
axs[1].plot(ts, xb, color=C["red"], lw=1.3, label="uncontrolled")
axs[1].plot(ts, xd, color=C["amber"], lw=1.3, ls="--", label="direct method")
axs[1].plot(ts, xn, color=C["blue"], lw=1.5, label="NN open-loop solver")
axs[1].set_ylabel(r"$\xi(t,1/2)$"); axs[1].legend(ncol=3, fontsize=8); axs[1].set_title("(b) midpoint displacement", fontsize=9)
axs[2].plot(ts, mu_d.numpy(), color=C["amber"], lw=1.2, ls="--", label="direct method")
axs[2].plot(ts, mu_nn.numpy(), color=C["blue"], lw=1.5, label="NN open-loop solver")
axs[2].axvspan(0, P2["kappa"] * tscale, color=C["grey"], alpha=0.2, lw=0)
axs[2].set_ylabel(r"$\mu(t)$"); axs[2].set_xlabel("physical time [s]"); axs[2].legend(fontsize=8)
axs[2].set_title(r"(c) optimal control (shaded: delay interval $[0,\kappa]$)", fontsize=9)
fig.tight_layout(); fig.savefig(OUT + "fig4_seismic_control.pdf"); fig.savefig(OUT + "fig4_seismic_control.png", dpi=200); plt.close(fig)

fig, axs = plt.subplots(1, 2, figsize=(7.2, 2.8))
axs[0].semilogy(h_nn, color=C["blue"], lw=1.4, label="NN open-loop solver")
axs[0].semilogy(h_d, color=C["amber"], lw=1.4, ls="--", label="direct method")
axs[0].axhline(J_unc, color=C["red"], lw=1.0, ls=":", label="uncontrolled")
axs[0].set_xlabel("iteration"); axs[0].set_ylabel(r"$\mathcal{J}(\mu)$"); axs[0].legend(fontsize=8); axs[0].set_title("(a) cost during training", fontsize=9)
axs[1].loglog(rob[:, 1], rob[:, 2], "o-", color=C["blue"], ms=4, lw=1.3, label=r"computed $\|\xi^{\varpi^1}-\xi^{\varpi^2}\|_\ast$")
xx = np.array([rob[:, 1].min(), rob[:, 1].max()])
axs[1].loglog(xx, TC2["Lambda_varpi"] * xx, color=C["red"], lw=1.3, ls="--", label=r"bound $\Lambda_\varpi\|\varpi^1-\varpi^2\|$")
axs[1].set_xlabel(r"$\|\varpi^1-\varpi^2\|_{L^2}$"); axs[1].legend(fontsize=8); axs[1].set_title("(b) robustness estimate", fontsize=9)
fig.tight_layout(); fig.savefig(OUT + "fig5_training_robustness.pdf"); fig.savefig(OUT + "fig5_training_robustness.png", dpi=200); plt.close(fig)

kap = {}
for k_ in [0.0, 0.1, 0.3]:
    Pk = dict(P2); Pk["kappa"] = k_; sk = Solver(Pk)
    _, Yk, _, Jk, _ = train_nn(sk, v1)
    kap[str(k_)] = dict(J=Jk, peak=float((Yk @ sk.Phi.T).abs().max()))
res["delay"] = kap; print("delay", kap)
v2 = torch.tensor(eq1 + 0.2 * (eq2 - eq1))
_, Y2, _, J2, _ = train_nn(sol2, v2)
res["value"] = dict(V1=J_nn, V2=J2, diff=abs(J_nn - J2), dw=l2t(0.2 * (eq2 - eq1), h2))
json.dump(res, open(OUT + "results_elcentro.json", "w"), indent=2)
np.savez(OUT + "elcentro_controls.npz", t=tg2, mu_nn=mu_nn.numpy(), mu_direct=mu_d.numpy(), hist_nn=np.array(h_nn), hist_direct=np.array(h_d))
print(json.dumps(res, indent=2))


In [ ]:
%%writefile project/code/fpinn_elcentro.py
"""fPINN for the uncontrolled structure under the recorded El Centro NS ground acceleration."""
import json, math, time, os
import numpy as np, torch, matplotlib, matplotlib.ticker
matplotlib.use("Agg")
import matplotlib.pyplot as plt
here = os.path.dirname(os.path.abspath(__file__))
src = open(os.path.join(here, "numerics_elcentro.py")).read()
__file__ = os.path.join(here, "numerics_elcentro.py")
exec(src[:src.index("v1 = torch.tensor(eq1)")])      # seismic setting, record, solver sol2

v1 = torch.tensor(eq1); M = sol2.M; Th = P2["Theta"]
with torch.no_grad():
    Yref, gref = sol2.solve(torch.zeros(M + 1), v1)

class Net(torch.nn.Module):
    """Small network for one time window, with a Fourier-feature input layer."""
    def __init__(s, N, width=64, nf=12, L=3.0):
        super().__init__()
        s.k = torch.arange(1, nf + 1).double() * 0.8; s.L = L
        s.f = torch.nn.Sequential(torch.nn.Linear(2 * nf + 1, width), torch.nn.Tanh(),
                                  torch.nn.Linear(width, width), torch.nn.Tanh(),
                                  torch.nn.Linear(width, width), torch.nn.Tanh(), torch.nn.Linear(width, N))
    def forward(s, r):                       # r = local time / L in [0,1]
        z = r * s.L
        return s.f(torch.cat([r, torch.sin(s.k * z), torch.cos(s.k * z)], 1))

# Time-marching fPINN: the interval is split into windows; each window has its own network,
# and the solution on earlier windows is frozen and enters through the memory term.
torch.manual_seed(1)
Aful = sol2.A.clone(); Aful[torch.arange(M + 1), torch.arange(M + 1)] = sol2.Ad
SC = 1e4; NW = 5; Lw = Th / NW; kw = int(round(Lw / P2["h"]))
tt = torch.tensor(tg2)
Yfix = torch.zeros(M + 1, P2["N"]); g_par = torch.zeros(P2["N"], requires_grad=True)
hist = []; it_glob = 0; t0 = time.time()
def assemble(net, j):
    s0, e0 = j * kw, (j + 1) * kw
    r = ((tt[s0:e0 + 1] - tt[s0]) / Lw).unsqueeze(1)
    if j == 0:
        y0 = sol2.phi_n + g_par
        Yw = y0 + r ** 2 * net(r)
    else:
        Yw = Yfix[s0] + r * net(r)
    Y = torch.cat([Yfix[:s0], Yw, Yw[-1:].expand(M - e0, -1)], 0)
    return Y, s0, e0
def loss_w(net, j):
    Y, s0, e0 = assemble(net, j)
    y0 = sol2.phi_n + g_par if j == 0 else (sol2.phi_n + g_par).detach()
    idx = torch.arange(M + 1) - sol2.dtau
    Ydel = torch.where((idx >= 0).unsqueeze(1), Y[idx.clamp(min=0)], y0)
    Jn = (-P2["zeta"] * torch.tanh(Ydel @ sol2.Phi.T)) @ sol2.Wq.T
    F = -sol2.om2 * Y + (Jn - sol2.m_n * v1.unsqueeze(1)) / sol2.E
    R = Y[s0:e0 + 1] - y0 - Aful[s0:e0 + 1] @ F
    l = torch.mean(torch.sum(R ** 2, 1)) / P2["N"]
    if s0 <= sol2.i1 <= e0:
        l = l + torch.sum((g_par - P2["c_g"] * Y[sol2.i1] / sol2.E) ** 2)
    return l, Y
for j in range(NW):
    net = Net(P2["N"], L=Lw)
    params = list(net.parameters()) + ([g_par] if j == 0 else [])
    opt = torch.optim.Adam(params, lr=2e-3)
    for it in range(8000):
        opt.zero_grad(); l, _ = loss_w(net, j); (SC * l).backward(); opt.step(); it_glob += 1
        if it == 5000:
            for gr in opt.param_groups: gr["lr"] = 5e-4
        if it_glob % 100 == 0: hist.append((it_glob, float(l.detach())))
    optL = torch.optim.LBFGS(params, max_iter=1500, tolerance_grad=1e-13, tolerance_change=1e-16,
                             history_size=50, line_search_fn="strong_wolfe")
    def closure():
        optL.zero_grad(); l, _ = loss_w(net, j); (SC * l).backward(); return SC * l
    for k in range(3):
        optL.step(closure); it_glob += 1500
        with torch.no_grad(): hist.append((it_glob, float(loss_w(net, j)[0])))
    with torch.no_grad():
        _, Y = loss_w(net, j); s0, e0 = j * kw, (j + 1) * kw
        Yfix[s0:e0 + 1] = Y[s0:e0 + 1]
    print("window", j, float(hist[-1][1]), time.time() - t0, flush=True)
Yp = Yfix.clone(); lval = hist[-1][1]
ttrain = time.time() - t0
xr = (Yref @ sol2.Phi.T).numpy(); xp = (Yp @ sol2.Phi.T).numpy()
err = np.abs(xr - xp)
res = dict(max_abs=float(err.max()), peak=float(np.abs(xr).max()), rel_l2=float(np.linalg.norm(xr - xp) / np.linalg.norm(xr)),
           g_err=float(torch.max(torch.abs(g_par - gref))), final_loss=float(lval), windows=NW, time=ttrain)
print(res)
json.dump(res, open(OUT + "results_fpinn_elcentro.json", "w"), indent=2)
np.savez(OUT + "fpinn_elcentro_data.npz", xr=xr, xp=xp, hist=np.array(hist))

iw = int(np.argmin(np.abs(S2["w"] - 0.5))); ts = tg2 * tscale
fig = plt.figure(figsize=(7.2, 4.9))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 1], hspace=0.5, wspace=0.45)
ax = fig.add_subplot(gs[0, :])
ax.plot(ts, xr[:, iw], color=C["blue"], lw=1.6, label="ABM (classical)")
ax.plot(ts[::3], xp[::3, iw], "o", ms=3.2, mfc="none", color=C["red"], label="fPINN")
ax.set_xlim(0, ts[-1]); ax.set_xlabel("physical time [s]"); ax.set_ylabel(r"$\xi(t,1/2)$")
ax.legend(ncol=2, fontsize=8, loc="lower left"); ax.set_title("(a) midpoint displacement under the El Centro record (no control)", fontsize=9)
ax2 = fig.add_subplot(gs[1, 0])
WW, TT = np.meshgrid(S2["w"], ts)
cs = ax2.contourf(TT, WW, 1e3 * err, 30, cmap="magma"); cb = fig.colorbar(cs, ax=ax2, fraction=0.05)
cb.locator = matplotlib.ticker.MaxNLocator(5); cb.update_ticks(); 
ax2.set_xlabel("physical time [s]"); ax2.set_ylabel(r"$w$"); ax2.grid(False)
ax2.set_title(r"(b) $|\xi_{\mathrm{ABM}}-\xi_{\mathrm{fPINN}}|\ (\times 10^{-3})$", fontsize=9)
ax3 = fig.add_subplot(gs[1, 1]); hl = np.array(hist)
ax3.semilogy(hl[:, 0], hl[:, 1], color=C["green"], lw=1.4)
ax3.set_xlabel("iteration"); ax3.set_ylabel("loss"); ax3.set_title("(c) fPINN training loss", fontsize=9)
fig.savefig(OUT + "fig_fpinn_elcentro.pdf", bbox_inches="tight"); fig.savefig(OUT + "fig_fpinn_elcentro.png", bbox_inches="tight", dpi=200)


In [ ]:
%%writefile project/code/export_data.py
"""Export all numerical data behind the tables and figures of the paper as CSV files.

Usage
    python export_data.py                 # tables + El Centro input + uncontrolled response + fPINN data (fast)
    python export_data.py --with-control  # also retrains the NN and direct controllers (about 10 min on a CPU)
                                          # and writes the controlled time histories

Output folder: ../data/processed/
"""
import os, sys, json, csv
import numpy as np, torch

here = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.abspath(os.path.join(here, ".."))
RES = os.path.join(ROOT, "results")
DST = os.path.join(ROOT, "data", "processed"); os.makedirs(DST, exist_ok=True)

def write(name, header, rows):
    with open(os.path.join(DST, name), "w", newline="") as f:
        w = csv.writer(f); w.writerow(header)
        for r in rows: w.writerow([f"{x:.10g}" if isinstance(x, (float, np.floating)) else x for x in r])
    print("wrote", name)

# ------------------------------------------------------------------ 1. tables from the JSON results
R1 = json.load(open(os.path.join(RES, "results.json")))              # smooth / synthetic study
R2 = json.load(open(os.path.join(RES, "results_elcentro.json")))     # El Centro study
R3 = json.load(open(os.path.join(RES, "results_fpinn_elcentro.json")))

rows = []
for k in ["varsigma", "Einv", "UpsE", "UpsP", "gamma1", "gamma2", "Dnorm", "Lambda_varpi"]:
    rows.append([k, R1["theory"][k], R2["theory"][k]])
write("table_theory_constants.csv", ["constant", "main_example_Theta4_zeta0.1", "elcentro_Theta15_zeta0.01"], rows)

f1 = R1["fpinn"]
write("table_fpinn_accuracy.csv",
      ["case", "rel_L2_error", "max_abs_error", "g_error", "final_loss", "peak_displacement"],
      [["smooth_forcing_single_network", f1["rel_l2"], f1["max_abs"], f1["g_err"], f1["final_loss"], f1["peak"]],
       ["elcentro_time_marching_5_windows", R3["rel_l2"], R3["max_abs"], R3["g_err"], R3["final_loss"], R3["peak"]]])

# table_order_peaks.csv and order_midpoint_curves.csv are written by fig3_order.py

oc = R2["oc"]
write("table_optimal_control_elcentro.csv", ["quantity", "uncontrolled", "NN_controller", "direct_method"],
      [["cost_J", oc["J_uncontrolled"], oc["J_nn"], oc["J_direct"]],
       ["peak_displacement", oc["peak_unc"], oc["peak_nn"], oc["peak_direct"]],
       ["rms_L2_norm", oc["rms_unc"], oc["rms_nn"], oc["rms_direct"]],
       ["max_abs_control", 0.0, oc["umax_nn"], oc["umax_direct"]],
       ["cpu_time_s", "", oc["time_nn"], oc["time_direct"]]])

write("table_control_delay_elcentro.csv", ["kappa", "optimal_cost_J", "peak_displacement"],
      [[float(k), v["J"], v["peak"]] for k, v in sorted(R2["delay"].items(), key=lambda z: float(z[0]))])

write("table_robustness_elcentro.csv",
      ["epsilon", "disturbance_gap_L2", "state_gap_sup", "ratio", "theoretical_bound_Lambda_times_gap"],
      [[e, d, s, s / d, R2["theory"]["Lambda_varpi"] * d] for e, d, s in R2["robustness"]["rows"]])

v = R2["value"]
write("table_value_function_elcentro.csv", ["V1", "V2", "abs_difference", "disturbance_gap_L2"],
      [[v["V1"], v["V2"], v["diff"], v["dw"]]])

# ------------------------------------------------------------------ 2. fPINN field data (El Centro)
d = np.load(os.path.join(RES, "fpinn_elcentro_data.npz"))
xr, xp, hist = d["xr"], d["xp"], d["hist"]

# ------------------------------------------------------------------ 3. El Centro setting (fast part)
src = open(os.path.join(here, "numerics_elcentro.py")).read()
__file__ = os.path.join(here, "numerics_elcentro.py")
exec(src[:src.index("v1 = torch.tensor(eq1)")])
ts = tg2 * tscale; wgrid = S2["w"]; iw = int(np.argmin(np.abs(wgrid - 0.5)))

write("elcentro_scaled_input.csv", ["t_model", "t_seconds", "ag_recorded_g", "varpi_model_units"],
      [[tg2[k], ts[k], eq1[k] * pga / PGA_ND, eq1[k]] for k in range(len(tg2))])

rows = []
for k in range(xr.shape[0]):
    for i in range(xr.shape[1]):
        rows.append([tg2[k], ts[k], wgrid[i], xr[k, i], xp[k, i], abs(xr[k, i] - xp[k, i])])
write("fpinn_elcentro_field.csv", ["t_model", "t_seconds", "w", "xi_ABM", "xi_fPINN", "abs_error"], rows)
write("fpinn_elcentro_training_loss.csv", ["iteration", "loss"], hist.tolist())

v1 = torch.tensor(eq1); mu0 = torch.zeros(M2 + 1)
with torch.no_grad():
    Yb, gb = sol2.solve(mu0, v1)
X = lambda Y: (Y @ sol2.Phi.T).numpy()
xb = X(Yb); nb = sol2.l2norm_t(Yb).numpy()

write("elcentro_uncontrolled_timeseries.csv", ["t_model", "t_seconds", "varpi", "xi_mid", "xi_L2norm"],
      [[tg2[k], ts[k], eq1[k], xb[k, iw], nb[k]] for k in range(len(tg2))])
if "--with-control" not in sys.argv:
    sys.exit(0)

# ------------------------------------------------------------------ 4. controlled responses (slow)
mu_nn, Y_nn, _, J_nn, h_nn = train_nn(sol2, v1)
mu_d, Y_d, _, J_d, h_d = train_direct(sol2, v1)
xn, xd = X(Y_nn), X(Y_d)
nn_, nd = sol2.l2norm_t(Y_nn).numpy(), sol2.l2norm_t(Y_d).numpy()
write("elcentro_control_timeseries.csv",
      ["t_model", "t_seconds", "varpi", "xi_mid_uncontrolled", "xi_mid_NN", "xi_mid_direct",
       "L2norm_uncontrolled", "L2norm_NN", "L2norm_direct", "mu_NN", "mu_direct"],
      [[tg2[k], ts[k], eq1[k], xb[k, iw], xn[k, iw], xd[k, iw], nb[k], nn_[k], nd[k],
        float(mu_nn[k]), float(mu_d[k])] for k in range(len(tg2))])
L = max(len(h_nn), len(h_d))
write("elcentro_control_training_cost.csv", ["iteration", "J_NN", "J_direct"],
      [[i + 1, h_nn[i] if i < len(h_nn) else "", h_d[i] if i < len(h_d) else ""] for i in range(L)])
print("J_NN", J_nn, "J_direct", J_d)


In [ ]:
%%writefile project/code/revision_checks.py
"""Additional numerical checks requested in the review (El Centro example).

1. Grid convergence of the ABM scheme in time (h) and in space (number of modes N).
2. Accuracy of the time-marching fPINN against a grid-converged reference solution.
3. Full-time-domain fPINN loss (Eq. (6.4)) of the assembled fPINN solution.
4. Effect of the nonlocal condition on the comparison of controlled and uncontrolled responses.
5. Time needed by the NN control solver and by the direct method to reach a target cost.

Run after numerics_elcentro.py and fpinn_elcentro.py. Output: ../results/revision_checks.json
Set RUN_LOCAL_IC = False to skip the extra optimization with c_g = 0 (about 6 minutes on a CPU).
"""
import json, math, os, time, csv
import numpy as np, torch

here = os.path.dirname(os.path.abspath(__file__))
src = open(os.path.join(here, "numerics_elcentro.py")).read()
__file__ = os.path.join(here, "numerics_elcentro.py")
exec(src[:src.index("v1 = torch.tensor(eq1)")])      # P2, sol2, record, tscale, train_direct, ...
ROOT = os.path.abspath(os.path.join(here, ".."))
RUN_LOCAL_IC = True
out = {}

def forcing(solver):
    """El Centro record on the time grid of a given solver (same scaling as in the paper)."""
    return torch.tensor(PGA_ND / pga * np.interp(solver.t * tscale, tr, ar, left=0.0, right=0.0))

def run(h=0.05, N=8, Nw=64, **kw):
    p = dict(P2); p.update(h=h, N=N, Nw=Nw); p.update(kw)
    s = Solver(p)
    with torch.no_grad():
        Y, g = s.solve(torch.zeros(s.M + 1), forcing(s), iters=60, tol=1e-13)
    return s, Y.numpy(), g.numpy()

# ------------------------------------------------------------------ 1a. convergence in time
t0 = time.time()
hs = [0.1, 0.05, 0.025, 0.0125, 0.00625, 0.003125]
sols = {h: run(h=h) for h in hs}
href = hs[-1]; _, Yref, _ = sols[href]
step = lambda h: int(round(0.1 / h))                      # common time points t = 0.1 k
Yr = Yref[::step(href)]
time_rows = []
for h in hs[:-1]:
    Y = sols[h][1][::step(h)]
    e_sup = np.max(np.linalg.norm(Y - Yr, axis=1)) / np.max(np.linalg.norm(Yr, axis=1))
    e_l2 = np.linalg.norm(Y - Yr) / np.linalg.norm(Yr)
    time_rows.append([h, h * tscale, float(e_sup), float(e_l2)])
for i in range(1, len(time_rows)):
    time_rows[i].append(math.log2(time_rows[i - 1][2] / time_rows[i][2]))
time_rows[0].append(float("nan"))
out["time_convergence"] = dict(columns=["h", "h_seconds", "rel_sup_error", "rel_L2_error", "observed_order"],
                               reference_h=href, rows=time_rows)
print("time convergence", time_rows, round(time.time() - t0, 1), flush=True)

# ------------------------------------------------------------------ 1b. convergence in space
Ns = [4, 8, 16, 32]
hsp = 0.0125
spc = {N: run(h=hsp, N=N, Nw=256) for N in Ns}
Yn_ref = spc[Ns[-1]][1]
space_rows = []
for N in Ns[:-1]:
    Y = np.zeros_like(Yn_ref); Y[:, :N] = spc[N][1]
    e_sup = np.max(np.linalg.norm(Y - Yn_ref, axis=1)) / np.max(np.linalg.norm(Yn_ref, axis=1))
    space_rows.append([N, float(e_sup)])
out["space_convergence"] = dict(columns=["N", "rel_sup_error_vs_N32"], h=hsp, rows=space_rows)
print("space convergence", space_rows, flush=True)

# ------------------------------------------------------------------ 2. fPINN against the converged reference
d = np.load(os.path.join(ROOT, "results", "fpinn_elcentro_data.npz"))
xp = d["xp"]                                             # fPINN on the h = 0.05 grid, 64 points
Wq = sol2.S["Wq"]
Yp = xp @ Wq.T                                           # modal coefficients (N = 8)
Yabm = sols[0.05][1]                                     # ABM with h = 0.05
Yfine = Yref[::int(round(0.05 / href))]                  # ABM with h = 0.003125, sampled at the same times
Yfine32 = np.zeros((Yfine.shape[0], 32))
_, Y32, _ = run(h=0.003125, N=32, Nw=256)
Yfine32 = Y32[::int(round(0.05 / 0.003125))]
pad = lambda Y: np.hstack([Y, np.zeros((Y.shape[0], 32 - Y.shape[1]))])
rel = lambda A, B: float(np.linalg.norm(A - B) / np.linalg.norm(B))
out["fpinn_accuracy"] = dict(
    fpinn_vs_abm_h005=rel(Yp, Yabm),
    fpinn_vs_abm_fine=rel(Yp, Yfine),
    abm_h005_vs_abm_fine=rel(Yabm, Yfine),
    fpinn_vs_converged_N32=rel(pad(Yp), Yfine32),
    abm_h005_vs_converged_N32=rel(pad(Yabm), Yfine32))
print("fPINN accuracy", out["fpinn_accuracy"], flush=True)

# ------------------------------------------------------------------ 3. full-time-domain fPINN loss
M = sol2.M
Aful = sol2.A.clone(); Aful[torch.arange(M + 1), torch.arange(M + 1)] = sol2.Ad
Ypt = torch.tensor(Yp); v1 = forcing(sol2)
y0 = Ypt[0]; g_p = y0 - sol2.phi_n
idx = torch.arange(M + 1) - sol2.dtau
Ydel = torch.where((idx >= 0).unsqueeze(1), Ypt[idx.clamp(min=0)], y0)
Jn = (-P2["zeta"] * torch.tanh(Ydel @ sol2.Phi.T)) @ sol2.Wq.T
F = -sol2.om2 * Ypt + (Jn - sol2.m_n * v1.unsqueeze(1)) / sol2.E
R = Ypt - y0 - Aful @ F
res_part = float(torch.mean(R ** 2)); nl_part = float(torch.sum((g_p - P2["c_g"] * Ypt[sol2.i1] / sol2.E) ** 2))
kw_ = M // 5
win = [float(torch.mean(R[j * kw_:(j + 1) * kw_ + 1] ** 2)) for j in range(5)]
out["fpinn_full_loss"] = dict(full_domain_loss=res_part + nl_part, residual_part=res_part, nonlocal_part=nl_part,
                              window_residuals=win, last_window_loss_reported=2.7893656025926794e-09)
print("full loss", out["fpinn_full_loss"], flush=True)

# ------------------------------------------------------------------ 4. nonlocal condition and control
_cz = os.path.join(ROOT, "results", "elcentro_controls.npz")
if os.path.exists(_cz):                                   # written by numerics_elcentro.py
    _c = np.load(_cz); mus = {"mu_NN": torch.tensor(_c["mu_nn"]), "mu_direct": torch.tensor(_c["mu_direct"])}
else:                                                     # data shipped with the repository
    rows = list(csv.DictReader(open(os.path.join(ROOT, "data", "processed", "elcentro_control_timeseries.csv"))))
    mus = {k: torch.tensor([float(r[k]) for r in rows]) for k in ["mu_NN", "mu_direct"]}
v1 = torch.tensor(eq1)
with torch.no_grad():
    Yb, gb = sol2.solve(torch.zeros(M + 1), v1)
    Jb = float(sol2.cost(Yb, torch.zeros(M + 1)))
    hist_norm = float(torch.linalg.norm(sol2.phi_n + gb))
    nl = dict(J_uncontrolled=Jb, history_norm_uncontrolled=hist_norm)
    for k, mu in mus.items():
        Y, g = sol2.solve(mu, v1)
        dk = sol2.dkap; u = torch.cat([torch.zeros(dk), mu[:M + 1 - dk]])
        Yfix_ = sol2.solve_given_g(gb, u, v1)              # same initial history as the uncontrolled case
        nl[k] = dict(J=float(sol2.cost(Y, mu)), J_same_history=float(sol2.cost(Yfix_, mu)),
                     rel_change_history=float(torch.linalg.norm(g - gb) / hist_norm),
                     reduction=1 - float(sol2.cost(Y, mu)) / Jb,
                     reduction_same_history=1 - float(sol2.cost(Yfix_, mu)) / Jb)
out["nonlocal_effect"] = nl
print("nonlocal", nl, flush=True)

if RUN_LOCAL_IC:
    p0 = dict(P2); p0["c_g"] = 0.0
    s0 = Solver(p0)
    with torch.no_grad():
        Y0, _ = s0.solve(torch.zeros(M + 1), v1); J0 = float(s0.cost(Y0, torch.zeros(M + 1)))
    t1_ = time.time()
    mu0, Yc0, gc0, Jc0, hc0 = train_direct(s0, v1)
    out["local_initial_condition"] = dict(c_g=0.0, J_uncontrolled=J0, J_direct=Jc0, reduction=1 - Jc0 / J0,
                                          peak_unc=float(np.abs(Y0.numpy() @ s0.S["Phi"].T).max()),
                                          peak_direct=float(np.abs(Yc0.numpy() @ s0.S["Phi"].T).max()),
                                          time=time.time() - t1_)
    print("c_g = 0", out["local_initial_condition"], flush=True)

# ------------------------------------------------------------------ 5. time to reach a target cost
if os.path.exists(_cz):
    hn, hd = _c["hist_nn"], _c["hist_direct"]
else:
    tc = list(csv.DictReader(open(os.path.join(ROOT, "data", "processed", "elcentro_control_training_cost.csv"))))
    hn = np.array([float(r["J_NN"]) for r in tc if r["J_NN"] != ""])
    hd = np.array([float(r["J_direct"]) for r in tc if r["J_direct"] != ""])
R0 = json.load(open(os.path.join(ROOT, "results", "results_elcentro.json")))["oc"]
spi_nn, spi_d = R0["time_nn"] / len(hn), R0["time_direct"] / len(hd)
Jbest = min(hn.min(), hd.min())
tt = []
for f in [1.10, 1.05, 1.02, 1.01]:
    target = f * Jbest
    i_nn = int(np.argmax(hn <= target)) + 1 if np.any(hn <= target) else None
    i_d = int(np.argmax(hd <= target)) + 1 if np.any(hd <= target) else None
    tt.append(dict(target_factor=f, target=target, iters_nn=i_nn, iters_direct=i_d,
                   time_nn=None if i_nn is None else i_nn * spi_nn, time_direct=None if i_d is None else i_d * spi_d))
out["time_to_target"] = dict(J_best=float(Jbest), sec_per_iter_nn=spi_nn, sec_per_iter_direct=spi_d, rows=tt)
print("time to target", out["time_to_target"], flush=True)

# ------------------------------------------------------------------ 6. computed controls on finer time grids
fine = {}
for hf in [0.05, 0.025, 0.0125]:
    pf = dict(P2); pf["h"] = hf; sf = Solver(pf); vf = forcing(sf)
    with torch.no_grad():
        Yu, _ = sf.solve(torch.zeros(sf.M + 1), vf); Ju = float(sf.cost(Yu, torch.zeros(sf.M + 1)))
        row = dict(J_uncontrolled=Ju, peak_unc=float(np.abs(Yu.numpy() @ sf.S["Phi"].T).max()))
        for k, mu in mus.items():
            muf = torch.tensor(np.interp(sf.t, sol2.t, mu.numpy()))
            Yc, _ = sf.solve(muf, vf)
            Jc = float(sf.cost(Yc, muf))
            row[k] = dict(J=Jc, reduction=1 - Jc / Ju, peak=float(np.abs(Yc.numpy() @ sf.S["Phi"].T).max()))
    fine[str(hf)] = row
out["controls_on_finer_grids"] = fine
print("finer grids", fine, flush=True)

json.dump(out, open(os.path.join(ROOT, "results", "revision_checks.json"), "w"), indent=2)
print("done")


## Step 3. Figure 1: the model problem

In [ ]:
!cd "{BASE}/code" && python fig0_model.py

## Step 4. Solver, theory constants and fPINN for the smooth forcing (≈18 min)
This step also draws early versions of Figures 4, 6, 7; they are replaced by Steps 5 and 6.

In [ ]:
!cd "{BASE}/code" && python numerics.py

## Step 5. Figure 4: effect of the fractional order (≈1 min)

In [ ]:
!cd "{BASE}/code" && python fig3_order.py

## Step 6. El Centro record: optimal control, delay effect, robustness (≈30 min)

In [ ]:
!cd "{BASE}/code" && python numerics_elcentro.py

## Step 7. Time-marching fPINN on the El Centro record (≈5 min)

In [ ]:
!cd "{BASE}/code" && python fpinn_elcentro.py

## Step 8. Export all numerical data as CSV (≈1 min)

In [ ]:
!cd "{BASE}/code" && python export_data.py

## Step 8b. Checks added in the revision (≈5 min)
Grid convergence in time and space, fPINN error against the converged reference, full-domain fPINN loss, effect of the nonlocal condition and time to reach a target cost (Section 6.6, Tables 5, 6 and 8).

In [ ]:
!cd "{BASE}/code" && python revision_checks.py

## Step 9. Show all figures

In [ ]:
from IPython.display import Image, display, Markdown
figs = [('Figure 1: model problem','fig0_model.png'),
        ('Figure 2: ABM solution and fPINN error','fig1_solution_error.png'),
        ('Figure 3: fPINN midpoint and training loss','fig2_fpinn.png'),
        ('Figure 4: effect of the order','fig3_order.png'),
        ('Figure 5: fPINN on El Centro','fig_fpinn_elcentro.png'),
        ('Figure 6: open-loop optimal control under El Centro','fig4_seismic_control.png'),
        ('Figure 7: training and robustness','fig5_training_robustness.png')]
for title, f in figs:
    path = os.path.join(BASE, 'results', f)
    if os.path.exists(path):
        display(Markdown('### ' + title))
        display(Image(filename=path, width=750))
    else:
        print('Not yet created:', f)

## Step 10. Main numerical results

In [ ]:
import json
R1 = json.load(open(os.path.join(BASE, 'results', 'results.json')))
R2 = json.load(open(os.path.join(BASE, 'results', 'results_elcentro.json')))
R3 = json.load(open(os.path.join(BASE, 'results', 'results_fpinn_elcentro.json')))
print('Theory constants (El Centro): gamma =', round(R2['theory']['gamma2'], 3), '| Lambda_varpi =', round(R2['theory']['Lambda_varpi'], 2))
print('fPINN rel. L2 error, smooth forcing :', round(100*R1['fpinn']['rel_l2'], 2), '%')
print('fPINN rel. L2 error, El Centro      :', round(100*R3['rel_l2'], 2), '%')
oc = R2['oc']
print('Cost: uncontrolled =', round(oc['J_uncontrolled'], 5), '| NN =', round(oc['J_nn'], 5), '| direct =', round(oc['J_direct'], 5))
print('Cost reduction by NN control:', round(100*(1 - oc['J_nn']/oc['J_uncontrolled']), 1), '%')
print('Peak displacement: uncontrolled =', round(oc['peak_unc'], 4), '| NN =', round(oc['peak_nn'], 4))
print('Delay effect (kappa: cost):', {k: round(v['J'], 5) for k, v in R2['delay'].items()})
print('Robustness: largest computed ratio =', round(R2['robustness']['empirical_gain'], 4), '< bound', round(R2['theory']['Lambda_varpi'], 2))

## Step 11. Save all results (figures, JSON and CSV) as one zip file
On Colab the zip downloads automatically. On Kaggle, open the **Output** panel on the right and download `results_seismic_control.zip`.

In [ ]:
import shutil
zip_path = shutil.make_archive(os.path.join(os.path.dirname(BASE), 'results_seismic_control'), 'zip', BASE, '.')
print('Saved:', zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    from IPython.display import FileLink, display
    display(FileLink(os.path.relpath(zip_path)))